# Fine Tuning Gemma2 2b for Hindi-Hinglish QnA

In this notebook we will walk through our step by step journey of fine tuning **Gemma2 2b model** into a conversational chatbot for Hindi.

This notebook will help you
  
  1. Understand the various parameters of crucial LoRA Layers. Where you will learn what the parameters do and how to tailor them for knowledge injection vs task tuning.

  2. How to effectively save your models after training and avoid unintentional inference issues.

  3. How to save resources while training as much as possible.

We used following datasets for our fine tuning.
- **GPT4's Alpaca**
- **Cognitive Lab's Hindi Instruct**
- **Wikipedia Datasets**
- **Alpaca for Gemma**
- **Databricks Dolly**
- **Hindi Maths Quest**

We experimented with different LoRA Configs in the various trainings with varying results depending on the configs

We mostly used **L4** and occasionaly **A100** GPUs as per our resource availability.

It took us about **40 Hours** to train on all the datasets, either full or a subset of them.

We learnt many things along the way specially related to the prompt for fine tuning and suitable LoRA Config for different tasks.

---


**All of our models and adapters that has been used/created in this project are available on kaggle**


You will need to place your kaggle token json file inside /root/.config/kaggle folder, you can download it from kaggle.

```python
import kagglehub

# Download latest version
path = kagglehub.model_download("lnshrivas/gemma-2/transformers/gemma-2-2b-hindi")

print("Path to model files:", path)
```

**Please pay attention to the `Notes` sections they are important to understand some crucial steps and concepts**

**Please check the `Understanding LoRA` section inside the *Conclusion* section before proceeding for enhancing your understanding about training from the getgo**

---

>DISCLAIMER - This Notebook was created as part of a Kaggle Competition

## 1. Inference Testing on base model

Let us test our base model's capacity for handling English, Hindi and Hinglish queries.

In [ ]:
!pip install -U bitsandbytes
!pip install datasets
!pip install trl
!pip install kaggle

In [ ]:
import torch
import warnings

from trl import SFTTrainer
from peft import PeftModel
from peft import LoraConfig
from tqdm.notebook import tqdm
from datasets import load_dataset

from transformers import BitsAndBytesConfig, TrainingArguments, AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq

We will download our base model first, from Kaggle.

You will also need a consent from google on kaggle to get this model. It's quite easy

https://www.kaggle.com/models/google/gemma-2/transformers/gemma-2-2b

In [ ]:
!kaggle models instances versions download google/gemma-2/transformers/gemma-2-2b/2

100% 9.05G/9.07G [00:48<00:00, 358MB/s]
100% 9.07G/9.07G [00:48<00:00, 200MB/s]
/content/gemma-2.tar.gz

We'll extract the model files

In [ ]:
!tar -xvzf 'gemma-2.tar.gz' 'gemma-2-2b'

---
>NOTE: We will load the model in 4-bit quantization for resource constraint training

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gemma-2-2b")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("gemma-2-2b",quantization_config=bnb_config,
                                                                         device_map='auto')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
question = "<start_of_turn>user Tell me about elephants, but tell me in English please. <end of turn>\n<start_of_turn>model "

inputs = tokenizer(question, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=246,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos><start_of_turn>user Tell me about elephants, but tell me in English please. <end of turn>
<start_of_turn>model 1. The elephant's name is Gungura. 2. She has light brown skin, small, dark brown tusks. Her eyes are a brown-black and she has no eyebrows, ears on top of her head with black tips, dark brown skin. Her trunk is long, with a darker trunk at the tip. Her long, white legs make the most of her body and help support her weight. The elephant has a wide, round, grey, trunk. Her trunk is long, but at the very end of the trunk has a dark-brown tip. Her trunk is long, but at the very end of the trunk has a dark-brown tip. (The trunk) is used for touching things with your nose and for breathing. Its long and thick, and it moves with your trunk. The elephant has long, grey, grey-blue, elephant ears, with black tips. This elephant has a wide, round, grey, elephant trunk, with dark-brown ears. It's a beautiful animal and a good friend, even if it is a little rough around the edges som

In [ ]:
question = "<start_of_turn>user recycling ke vishay me ek nara sujhav kare<end of turn>\n<start_of_turn>model "

inputs = tokenizer(question, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=246,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos><start_of_turn>user recycling ke vishay me ek nara sujhav kare<end of turn>
<start_of_turn>model 2/8
ServletConfig config = new ServletConfig();

	config.setServletName("web.xml");

	ServletContext servletContext = config.getServletContext();
	servletContext.setSessionTrackingPolicy(Servlet.SESSION_MODE_COOKIE);

	ServletRegistrationConfig config2 = new ServletRegistrationConfig();

	config2.setServletName("ServletRegistrationConfig");

	config2.setInitParameter("name","web.xml");
	

	ServletRegistrationConfig config3 = new ServletRegistrationConfig();

	config3.setInitParameter("name","myServlet");
		


	SessionConfig config4 = new SessionConfig();

		ServletRegistrationConfig config5 = new ServletRegistrationConfig();
	
		config5.setServletName("ServletRegistrationConfig");
	

	ServletContext servletContext2 = config2.getServletContext();
	

	servletContext2.setSessionTrackingPolicy(SessionTrackingPolicy.COOKIE);
	

	ServletContext servletContext3 = config3.getServletContext();


In [ ]:
question = "<start_of_turn>user रीसाइक्लिंग के विषय में एक नारा सुझाए<end of turn>\n<start_of_turn>model "

inputs = tokenizer(question, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=246,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos><start_of_turn>user रीसाइक्लिंग के विषय में एक नारा सुझाए<end of turn>
<start_of_turn>model 550-year-old 80-year-old man, 2006 and 1911.

The man is a citizen of the city of 80 years of old. He came to the hospital in a wheelchair to the hospital after a car accident.
It is diagnosed and treated by the hospital.
It has been discovered that the patient's bone density is reduced by 50% and has a risk of bone fracture.
It has to take the medication for life.
It is recommended by the doctor that this man take the drug daily for a long time.
The patient agrees to this.
But, for some reason, he stops taking his medication.
He falls on the ground after taking his medication.
He was sent to the hospital again.

In the hospital, the doctor saw that the patient's condition is getting worse.
After being told the patient that he could only make it through two nights, I immediately called the family to get here.
The doctor is ready to send the patient home for hospice care.

After the patient 

As you can see it cannot handle hinglish nor hindi queries, Generating garbled output.

---

## 2. Fine tuning on Alpaca Dataset and making it understand Hindi

We trained for a total of **15 Hrs** on this dataset

### Dataset Preparation

> NOTE: These settings often help in reducing reserved GPU memory to increase available memory and can help in save resources for training.

In [ ]:
!export TORCH_CUDA_ALLOC_CONF=max_split_size_mb:128

In [ ]:
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
alpaca_dataset_train = load_dataset("FreedomIntelligence/alpaca-gpt4-hindi",
                              split = "train")
alpaca_dataset_train, alpaca_dataset_train[3]

Repo card metadata block was not found. Setting CardData to empty.


(Dataset({
     features: ['conversations', 'id'],
     num_rows: 49969
 }),
 {'conversations': [{'from': 'human',
    'value': 'विज्ञान शिक्षा के महत्व पर 100 शब्दों का एक पैराग्राफ लिखें।\n'},
   {'from': 'gpt',
    'value': 'विज्ञान शिक्षा संवेदनशील सोच, नवाचार और प्रौद्योगिकी के लिए आधार रखती है। इससे छात्र अपने आसपास की दुनिया को समझते हैं और पूछताछ, जिज्ञासा और तार्किक तर्क का एक भाव विकसित करते हैं। विज्ञान शिक्षा के माध्यम से छात्र समस्या का समाधान करना सीखते हैं, प्राकृतिक घटनाओं का अध्ययन करते हैं और पर्यावरण पर मनुष्य के कार्यों का प्रभाव का अध्ययन करते हैं। इसके अलावा, विज्ञान शिक्षा विज्ञान, प्रौद्योगिकी, इंजीनियरिंग और गणित (STEM) क्षेत्रों में कैरियर के लिए छात्रों को तैयार करती है जो सामाजिक और आर्थिक प्रगति के लिए ब्रेकथ्रू उत्पन्न करते हैं। इसलिए, विज्ञान शिक्षा किसी भी शैक्षिक प्रणाली का एक महत्वपूर्ण घटक है और एक राष्ट्र के लिए आवश्यक है जो वैश्विक मंच पर प्रतिस्पर्धात्मक बना रहना चाहता है।'}],
  'id': '10608'})

In [ ]:
alpaca_dataset_train.info

DatasetInfo(description='', citation='', homepage='', license='', features={'conversations': [{'from': Value(dtype='string', id=None), 'value': Value(dtype='string', id=None)}], 'id': Value(dtype='string', id=None)}, post_processed=None, supervised_keys=None, builder_name='json', dataset_name='alpaca-gpt4-hindi', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=94243803, num_examples=49969, shard_lengths=None, dataset_name='alpaca-gpt4-hindi')}, download_checksums={'hf://datasets/FreedomIntelligence/alpaca-gpt4-hindi@fcd2e9a085696cf16a040df158eb31d1cd74c212/alpaca-gpt4-hindi.json': {'num_bytes': 101647883, 'checksum': None}}, download_size=101647883, post_processing_size=None, dataset_size=94243803, size_in_bytes=195891686)

In [ ]:
alpaca_prompt="""<start_of_turn>user\n.\n\"{}\"<end_of_turn>\n<start_of_turn>model\n{}<end_of_turn>"""
print(alpaca_prompt)

<start_of_turn>user
.
"{}"<end_of_turn>
<start_of_turn>model
{}<end_of_turn>


---

> NOTE: We have set the padding to right. This will ensure the padding will be added to the right of the tokens. Without this tokenizer may pad to the left.

In [ ]:
eos_token = tokenizer.eos_token
tokenizer.padding_side = "right"
eos_token

'<eos>'

In [ ]:
def formatting_func(conversations):
    texts = []
    conversations = conversations["conversations"]
    for convo in conversations:
        # EOS_TOKEN is important
        text = alpaca_prompt.format(convo[0]["value"], convo[1]["value"]) + eos_token
        texts.append(text)
    return { "text" : texts, }

In [ ]:
alpaca_dataset = alpaca_dataset_train.map(formatting_func, batched = True,)

In [ ]:
alpaca_dataset

Dataset({
    features: ['conversations', 'id', 'text'],
    num_rows: 49969
})

---

> NOTE: This prompt will help our model understand the user query and model generation part. The eos (end of sentence) token is very important without which the model will use to know when to stop regardless of max seq length to generate otherwise it will generate endless tokens which we will see happening in action, ahead

In [ ]:
print(alpaca_dataset["text"][0])

<start_of_turn>user
.
"कुछ एक रीसाइक्लिंग अभियान के लिए एक नारा सुझाव दें।
"<end_of_turn>
<start_of_turn>model
1. "ग्रीन भविष्य के लिए एक साथ: कम करें, पुन: उपयोग करें, रीसाइकल करें।"
2. "एक बेहतर कल के लिए आज ही रीसाइकल करें।"
3. "अपने कचरे को खजाना बनाएं - रीसाइकल करें!"
4. "जीवन के चक्र के लिए रीसाइकल करें।"
5. "संसाधन बचाएं, अधिक रीसाइकल करें।"<end_of_turn><eos>


---

> NOTE: This function converts the generated prompt text into tokens along with padding, and also generates attention masks for our tokens, telling the model which tokens to pay attention to and ignore the padding tokens

In [ ]:
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

print("Tokenizing dataset...")
dataset = alpaca_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print("Dataset tokenized:", dataset[0])

Tokenizing dataset...
Dataset tokenized: {'conversations': [{'from': 'human', 'value': 'कुछ एक रीसाइक्लिंग अभियान के लिए एक नारा सुझाव दें।\n'}, {'from': 'gpt', 'value': '1. "ग्रीन भविष्य के लिए एक साथ: कम करें, पुन: उपयोग करें, रीसाइकल करें।"\n2. "एक बेहतर कल के लिए आज ही रीसाइकल करें।"\n3. "अपने कचरे को खजाना बनाएं - रीसाइकल करें!"\n4. "जीवन के चक्र के लिए रीसाइकल करें।"\n5. "संसाधन बचाएं, अधिक रीसाइकल करें।"'}], 'id': '23712', 'input_ids': [2, 106, 1645, 108, 235265, 108, 235281, 132527, 236974, 15848, 9184, 210123, 33608, 17640, 235620, 42828, 84735, 24081, 235530, 6777, 19126, 15848, 7578, 38266, 53474, 238385, 40936, 188793, 235940, 108, 235281, 107, 108, 106, 2516, 108, 235274, 235265, 664, 164624, 235530, 12076, 38311, 212133, 6777, 19126, 15848, 36631, 235292, 72792, 38607, 235269, 163169, 235292, 86467, 38607, 235269, 9184, 210123, 33608, 144122, 38607, 235940, 235281, 108, 235284, 235265, 664, 124025, 193899, 103217, 2280, 235620, 6777, 19126, 133533, 40778, 9184, 210123, 33

---
### Training

> NOTE: This LoRA Config helped us generalize faster with our dataset, however if you wish to inject knowledge to ur model
- keep **"r"** high
- remove the feed forward layers from target modules
- use_rslora=False

> However in our implementation we didn't follow through it.

> To save resources while training
- use smaller batch size
- keep gradient accumulation to 1
- save limit to 1 or 2
- set a torch empty cache parameter other wise after long runs the build up of cache will crash the GPU

> This will help keep the GPU Memory requirements under 20 GBs. Suitable for L4 GPU

In [ ]:
lora_config = LoraConfig(
    r=128,
    lora_alpha=256,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["embed_tokens", "lm_head"],
    task_type="CAUSAL_LM",
    use_rslora=True
)

train_args = TrainingArguments(
    per_device_train_batch_size=2,  # Each GPU processes 2 examples per step.
    gradient_accumulation_steps=1,  # Gradients are accumulated over 1 steps before updating weights.
    # warmup_steps=30,  # Learning rate warms up (gradually increases) for the first 30 steps.
    #max_steps=10,  # Total number of optimization steps for training.
    warmup_ratio=0.1, # Learning rate warms up (gradually increases) for the first 10 percent of epoch.
    num_train_epochs=1,  # Total number of epochs for training.
    gradient_checkpointing=True,  # Saves memory by recomputing activations during backpropagation.
    learning_rate=5e-5,  # Base learning rate for the optimizer.
    fp16=not torch.cuda.is_bf16_supported(),  # FP16 precision if BF16 is not available.
    bf16=torch.cuda.is_bf16_supported(),  # Enables bfloat16 precision if available.
    save_steps=100,  # Saves checkpoint every 100 steps.
    torch_empty_cache_steps = 100,  # Empties the cache at every 100 steps.
    optim="adamw_8bit",  # Uses AdamW optimizer with 8-bit precision for optimizer states to save memory.
    weight_decay=0.01,  # Regularization to prevent overfitting by penalizing large weights.
    lr_scheduler_type="linear",  # Linearly decays learning rate after the warmup period.
    output_dir="gemma-2-2b-{hi)-alpaca-chk",  # Directory where model checkpoints and logs will be saved.
    report_to="none",  # Disables logging to external tools like TensorBoard or WandB.
    save_total_limit=2, # Will save only 2 checkpoints at max, reducing the disk usage.
    run_name='pretrain_gemma2' # Defining a name for our runtime.
)

---
> NOTE: In both tokenizer and collator some arguments are commmon.
- **padding** 'longest' to find the longest batch and pad based on that
- **padding** 'max_length' will then require u to set a
  - **max_length** parameter for padding,
  - **truncation** parameter to cut any sequences longer than max_length

In [ ]:
# If you did not tokenized the dataset, you must use Data Collator.
# It uses tokenizer, tokenize your training data and returns them as tensors.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model,
    padding="longest",
    return_tensors="pt"
)

trainer = SFTTrainer(
    model=base_model,
    tokenizer=tokenizer,
    args=train_args,
    peft_config=lora_config,
    train_dataset=dataset,
    data_collator=data_collator,
)

<ipython-input-14-2ae35be4ed0d>:12: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


In [ ]:
# To begin training use
trainer.train()

Step,Training Loss
500,1.053100
1000,0.502300
1500,0.560300
2000,0.585600
2500,0.608800
3000,0.674400
3500,0.681200
4000,0.649900
4500,0.659100
5000,0.661200


In [ ]:
# To resume training from last checkpoint use
trainer.train(resume_from_checkpoint=True)

In [ ]:
# Once training is done save the model and the tokenizer
trainer.save_model('gemma-2-2b-(hi)-24985steps-1epoch-alphacha')
trainer.tokenzier.save_pretrained('gemma-2-2b-(hi)-24985steps-1epoch-alphacha')

> NOTE: The training loss as you can see gets low quickly and can go even lower as we training it for more epochs

---
### Inference

For inference you will need to merge the base model and adapter model like this

> NOTE: Always merge a non quantized base model with the adapter to avoid rounding errors during inference and causing unexpected behaviour

In [ ]:
model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained('gemma-2-2b', device_map="cpu"), 'gemma-2-2b-24985steps-1epoch-alphacha')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
question = "कुछ एक रीसाइक्लिंग अभियान के लिए एक नारा सुझाव दें।"



inputs = tokenizer(question, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=128,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

कुछ एक रीसाइक्लिंग अभियान के लिए एक नारा सुझाव दें।
"
model
1. "एकल उपयोग पर दोबारा उपयोग करने के लिए, केवल रीसाइक्ल करें"
2. "हर कदम से शुरू करें, हर भोजन में बदलें"
3. "आवश्यक सामग्री के लिए प्लास्टिक से बने प्लास्टिक की मात्रा को कम करना"
4. "अपनी उत्पादन को सुधारते हुए प्लास्टिक के उपभोग को कम करना"
5. "अनावश्यक सामग्री की एकल उपयोग से दोबारा उपयोग करें, फिर से उपयोग करें"


You can see how the model is now able to answer to a hindi query that according to the dataset it has learned from

---

### Saving the model as transformer file

> NOTE: These steps are very curcial for ensuring the model weights are properly transfered as without this we faced an inference loss where our saved pretrained model wasn't able to infer properly without the state dictionary weight transfer so we made it a default saving frame work for us

In [ ]:
merged_model.save_pretrained("gemma-2-2b-tmp")

In [ ]:
torch.save(merged_model.state_dict(), "merged_model_state_dict.pth")

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gemma-2-2b-tmp",device_map='cpu')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
model.load_state_dict(torch.load("merged_model_state_dict.pth", weights_only=True))

<All keys matched successfully>

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gemma-2-2b-{hi)-24985steps-1epoch-alphacha")

In [ ]:
model.save_pretrained("gemma-2-2b-base+alpaca")

In [ ]:
tokenizer.save_pretrained("gemma-2-2b-base+alpaca")

Saved model inference -

In [ ]:
question = "कुछ एक रीसाइक्लिंग अभियान के लिए एक नारा सुझाव दें।"

inputs = tokenizer(question, return_tensors="pt").to('cpu')

generated_ids = model.generate(**inputs,
                              max_new_tokens=128,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

कुछ एक रीसाइक्लिंग अभियान के लिए एक नारा सुझाव दें।"
model
"एक सचमुच रीसाइक्लिंग अभियान के लिए एक नारा सुझाव है "हर बार रीसाइक्ल करें - हर बार रीसाइकल करें!"


---
## 3. Fine tuning on Congitive Lab Hindi Instruct Dataset

We decided to further fine tune our model on another dataset to test what kind of changes will that bring to our model and possibly increase its knowledge base further (it didn't go very well)

Here's where we learnt the importance of definining the perfect LoRA Configs to train or fine tune any LLM

We trained for a total of **20 Hrs** on this dataset

> NOTE: We will always load a quantized model for training.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gemma-2-2b-base+alpaca")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("gemma-2-2b-base+alpaca",quantization_config=bnb_config,
                                                                         device_map='auto')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

---
> NOTE: In this data set as we can see they tried to help the model understand the previous user-model interactions which can be useful if you want to instruction tune it to understand how to use previous conversations when you are using History in an LLM

> However there is a major issue in the training prompt. Can you see it? We accidentally trained our model on this dataset without rectifying the issue which led to a big problem during inference which we will see ahead

> We will also delve into ways of resolving that issue in a lot cheaper way

### Dataset Preparation

In [ ]:
cognitive_hi_inst_train = load_dataset("CognitiveLab/Hindi-Instruct", split='train')
cognitive_hi_inst_test = load_dataset("CognitiveLab/Hindi-Instruct", split='test')

In [ ]:
cognitive_hi_inst_train, print(cognitive_hi_inst_train[0]['text']), print(cognitive_hi_inst_train[0]['input_ids'])

<start_of_turn>user Tell me everything you know about Hitesh Mehta, but tell me in Hindi please. <end_of_turn>
<start_of_turn>model हितेश मेहता का जन्म 1959 में नैरोबी, केन्या में हुआ था। वह एक भारतीय परिवार से संबंधित थे। हितेश एक आर्किटेक्ट और एक प्रभावशाली क्रिकेट खिलाड़ी थे। 1979 और 1980 के शुरुआती वर्षों में, हितेश ने पूर्वी अफ्रीका और केन्या के लिए क्रिकेट मैच खेले। उन्होंने ICC ट्रॉफी में केन्या के लिए 12 मैच खेले। साथ ही, उन्होंने 1986 में केन्या के खिलाफ पाकिस्तान स्टारलेट्स के खिलाफ एक महत्वपूर्ण मैच खेला जिसमें उन्होंने कुल 56 रन बनाए और स्टारलेट्स की पहली पारी में तीन विकेट लिए। क्रिकेट के अलावा, हितेश को 'ईको-आर्किटेक्ट' के रूप में भी जाना जाता है। वे 2006 में नेशनल जियोग्राफिक एडवेंचर द्वारा सस्टेनेबल टूरिज्म में पांच वैश्विक अगुआओं में से एक के रूप में मान्यता प्राप्त हुई थी। उन्होंने 2010 में 'ऑथेंटिक ईकोलॉज' नामक पुस्तक लिखी थी जिसका प्रकाशन HarperCollins ने किया था। <end_of_turn>
<start_of_turn>user क्या वह अब भी क्रिकेट खेलते हैं? <end_of_turn>
<start_of_turn>model इ

(Dataset({
     features: ['messages', 'prompt', 'text', 'input_ids', 'attention_mask'],
     num_rows: 91328
 }),
 None,
 None)

This tokenizer is more or less the same as we used for the other dataset

In [ ]:
def tokenize_function(examples):
    tokenizer.padding_side = "right"
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

print("Tokenizing dataset...")
train_dataset = cognitive_hi_inst_train.map(tokenize_function, batched=True, remove_columns=["text"])
test_dataset = cognitive_hi_inst_test.map(tokenize_function, batched=True, remove_columns=["text"])
print("Dataset tokenized:", train_dataset[0])

Tokenizing dataset...


Map:   0%|          | 0/91328 [00:00<?, ? examples/s]

Map:   0%|          | 0/10148 [00:00<?, ? examples/s]

Dataset tokenized: {'messages': [{'content': 'Tell me everything you know about Hitesh Mehta, but tell me in Hindi please.', 'role': 'user'}, {'content': "हितेश मेहता का जन्म 1959 में नैरोबी, केन्या में हुआ था। वह एक भारतीय परिवार से संबंधित थे। हितेश एक आर्किटेक्ट और एक प्रभावशाली क्रिकेट खिलाड़ी थे। 1979 और 1980 के शुरुआती वर्षों में, हितेश ने पूर्वी अफ्रीका और केन्या के लिए क्रिकेट मैच खेले। उन्होंने ICC ट्रॉफी में केन्या के लिए 12 मैच खेले। साथ ही, उन्होंने 1986 में केन्या के खिलाफ पाकिस्तान स्टारलेट्स के खिलाफ एक महत्वपूर्ण मैच खेला जिसमें उन्होंने कुल 56 रन बनाए और स्टारलेट्स की पहली पारी में तीन विकेट लिए। क्रिकेट के अलावा, हितेश को 'ईको-आर्किटेक्ट' के रूप में भी जाना जाता है। वे 2006 में नेशनल जियोग्राफिक एडवेंचर द्वारा सस्टेनेबल टूरिज्म में पांच वैश्विक अगुआओं में से एक के रूप में मान्यता प्राप्त हुई थी। उन्होंने 2010 में 'ऑथेंटिक ईकोलॉज' नामक पुस्तक लिखी थी जिसका प्रकाशन HarperCollins ने किया था।", 'role': 'assistant'}, {'content': 'क्या वह अब भी क्रिकेट खेलते हैं?', 'role': 

### Training

---
> NOTE: This time we decided to
- only focus on the attention and feed forward layers and
- exclude the embedding and lm layers while training.

> We also reduced
- **"r"** and
- lora_alpha
- torch empty cache parameter to save more vram for larger batch size

> We will see its consequences ahead

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=128,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj","gate_proj", "up_proj", "down_proj"
                    ],
    #modules_to_save=["embed_tokens", "lm_head"],
    task_type="CAUSAL_LM",
    use_rslora=True
)

train_args = TrainingArguments(
    per_device_train_batch_size=3,  # Each GPU processes 4 examples per step.
    gradient_accumulation_steps=1,  # Gradients are accumulated over 4 steps before updating weights.
    # warmup_steps=30,  # Learning rate warms up (gradually increases) for the first 30 steps.
    #max_steps=10,  # Total number of optimization steps for training.
    warmup_ratio=0.1, # Learning rate warms up (gradually increases) for the first 10 percent of epoch.
    num_train_epochs=1,  # Total number of epochs for training.
    gradient_checkpointing=True,  # Saves memory by recomputing activations during backpropagation.
    learning_rate=5e-5,  # Base learning rate for the optimizer.
    fp16=not torch.cuda.is_bf16_supported(),  # FP16 precision if BF16 is not available.
    bf16=torch.cuda.is_bf16_supported(),  # Enables bfloat16 precision if available.
    save_steps=100,  # Saves checkpoint every 100 steps.
    torch_empty_cache_steps=10,  # Empties the cache at every 10 steps.
    optim="adamw_8bit",  # Uses AdamW optimizer with 8-bit precision for optimizer states to save memory.
    weight_decay=0.01,  # Regularization to prevent overfitting by penalizing large weights.
    lr_scheduler_type="linear",  # Linearly decays learning rate after the warmup period.
    output_dir="gemma-2-2b-cog-lab-chk",  # Directory where model checkpoints and logs will be saved.
    report_to="none",  # Disables logging to external tools like TensorBoard or WandB.
    save_total_limit=2, # Will save only 2 checkpoints at max, reducing the disk usage.
    run_name='pretrain_gemma2' # Defining a name for our runtime.
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model,
    padding="longest",
    return_tensors="pt"
)

trainer = SFTTrainer(
    model=base_model,
    tokenizer=tokenizer,
    args=train_args,
    peft_config=lora_config,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

<ipython-input-17-5dcaa33d1df3>:12: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:359: UserWarning: You passed a dataset that is already processed (contains an `input_ids` field) together with a valid formatting function. Therefore `formatting_func` will be ignored. Either remove the `formatting_func` or pass a dataset that is not already processed.
  warnings.warn(


---
> NOTE: Notice now how the training loss didn't change too drastically unlike the last time. This can be the result of the LoRA Config, some hyper parameters or the dataset itself

In [ ]:
trainer.train()

Step,Training Loss
500,2.105500
1000,1.761700
1500,1.632700
2000,1.597300
2500,1.572100
3000,1.517900
3500,1.522900
4000,1.482500
4500,1.491200
5000,1.442100


Follow the exact same saving steps

In [ ]:
trainer.save_model("gemma-2-2b-30443steps-1epoch-cog-lab")

In [ ]:
trainer.tokenizer.save_pretrained('gemma-2-2b-30443steps-1epoch-cog-lab')

### Inference

Load the merge the base and adapter

In [ ]:
merged_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained("gemma-2-2b-base+alpaca",device_map='cpu'), 'gemma-2-2b-30443steps-1epoch-cog-lab').merge_and_unload()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
question = "<start_of_turn>user क्या आप मुझे रीसाइक्लिंग के लिए एक नारा समझा सकते हैं? <end of turn>\n<start_of_turn>model "


inputs = tokenizer(question, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=246,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos><start_of_turn>user क्या आप मुझे रीसाइक्लिंग के लिए एक नारा समझा सकते हैं? <end of turn>
<start_of_turn>model 24/30 भारत की अर्थव्यवस्था के सबसे उन्नत परियोजनाओं में से एक है, और इसने सततता और पर्यावरणीय जिम्मेदारी के प्रति जागरूकता बढ़ाने में महत्वपूर्ण भूमिका निभाई है। नया आयात, 2022 में मनाया गया, इस परियोजना की व्यावहारिक गतिशीलता को देखते हुए एक नया जीवन शुरू कर दिया। इस परियोजना का एक उदाहरण स्वच्छ ईंधन और गैर-स्वच्छ ईंधन में परिवर्तन है।

यह परियोजना नौकरियों और आर्थिक विकास को बढ़ावा देने में भी महत्वपूर्ण भूमिका निभाती है। उदाहरण के लिए, फर्नीचर और वस्तुओं को स्थापित करने, ईंधन की बचत करने और स्थायी रूप से प्राप्त उत्पादों को पुनर्परिष्कार करने की प्रक्रिया का एक महत्वपूर्ण पहलू है। एक अन्य अनुसंधान 'पावर पाइपिंग' में बढ़ती हुई जागरूकता है, जिसका उद्देश्य वित्तीय रूप से


---
> NOTE: Notice how the model is unable to stop its generation unlike the last time. Can you guess why this is happening?

> That's right, the end of sentence token. If you notice in our training prompt there was no `<eos>` token. This means the model didn't learn when to stop it's generation and keeps on continuing with more tokens.

> To fix this we simply need to add the `<bos>` and `<eos>` tokens to our prompt to tell the model what is the begenning and the end of a conversation

### Saving the model

Ofc first we will save our model like before

In [ ]:
merged_model.save_pretrained("gemma-2-2b-tmp")

In [ ]:
torch.save(merged_model.state_dict(), "merged_model_state_dict.pth")

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gemma-2-2b-tmp",device_map='cpu')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
model.load_state_dict(torch.load("merged_model_state_dict.pth", weights_only=True))

<All keys matched successfully>

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gemma-2-2b-30443steps-1epoch-cog-lab")

In [ ]:
model.save_pretrained("gemma-2-2b-base+alpaca+cog-lab")

In [ ]:
tokenizer.save_pretrained("gemma-2-2b-base+alpaca+cog-lab")

### Further Fine Tuning to teach beginning and end tokens

In [ ]:
def format_prompt_with_tokens(batch):
    formatted_prompts = []
    for conversations in batch['messages']:
        formatted_prompt = []
        for i in range(0, len(conversations), 2):  # Process user-model pairs
            user_message = conversations[i]
            model_message = conversations[i + 1] if i + 1 < len(conversations) else None

            if user_message['role'] == "user" and model_message and model_message['role'] == "assistant":
                formatted_prompt.append(
                    f"<bos><start_of_turn>{user_message['role']} {user_message['content']} <end_of_turn>\n"
                    f"<start_of_turn>model {model_message['content']} <end_of_turn><eos>"
                )

        # Join the formatted prompt for this conversation
        formatted_prompts.append("\n".join(formatted_prompt))

    # Return the formatted text as a new field in the dataset
    return {"text": formatted_prompts}

---
> NOTE: This time we take 3000 samples for fine tuning the model to teach it when to use `<eos>` token

In [ ]:
# Shuffle the dataset and take 1000 examples
random_subset = cognitive_hi_inst_train.take(3000)

# Apply the formatting function to this subset
cognitive_hi_inst_dataset = random_subset.map(format_prompt_with_tokens, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
print(cognitive_hi_inst_dataset[1]['text'])

<bos><start_of_turn>user Can you tell me about the plot of the film Vartul, but in Hindi please? <end_of_turn>
<start_of_turn>model वर्तुल एक छोटी फिल्म है जो 10 साल के एक लड़के दामू के बारे में है। उसकी माँ एक त्योहार के लिए मिठाई बनाना चाहती है और दामू को दुकान से गुड़ खरीदने के लिए भेजती है। हालांकि, वह नोटिस करता है कि पैसे के डिब्बे में सिर्फ एक सिक्का बचा है। दुकान की तरफ जाते समय, दामू अपने दोस्त गण्या के साथ दौड़ता है और वे एक शो देखने के लिए रुकते हैं। चूंकि वह शो देखने के लिए नहीं खरीद सकता, वह दुकान जाकर गुड़ खरीदता है। शो के बाद, गण्या उन्हें इसके बारे में बताता है जब वे अपनी बाइक को सवारी करते हैं।

एक जुआ खेल द्वारा आकर्षित, जहां आप एक सिक्के पर पत्थर का स्लैब फेंकते हैं, दामू कुछ पैसे कमाने का फैसला करता है। वह शुरू में जीतता है, लेकिन जब वह दुकान जाता है, दुकानदार उसे बताता है कि उसने जो सिक्का इस्तेमाल किया था, वह मान्य नहीं है। परेशान, दामू रोते हुए घर चला जाता है। <end_of_turn><eos>
<bos><start_of_turn>user फिल्म की निर्माण प्रक्रिया के बारे में बताइए। मेरा उत्तर हिं

In [ ]:
def tokenize_function(examples):
    tokenizer.padding_side = "right"
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

print("Tokenizing dataset...")
train_dataset = cognitive_hi_inst_dataset.map(tokenize_function, batched=True)
print("Dataset tokenized:", train_dataset[0])

Tokenizing dataset...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset tokenized: {'messages': [{'content': 'Tell me everything you know about Hitesh Mehta, but tell me in Hindi please.', 'role': 'user'}, {'content': "हितेश मेहता का जन्म 1959 में नैरोबी, केन्या में हुआ था। वह एक भारतीय परिवार से संबंधित थे। हितेश एक आर्किटेक्ट और एक प्रभावशाली क्रिकेट खिलाड़ी थे। 1979 और 1980 के शुरुआती वर्षों में, हितेश ने पूर्वी अफ्रीका और केन्या के लिए क्रिकेट मैच खेले। उन्होंने ICC ट्रॉफी में केन्या के लिए 12 मैच खेले। साथ ही, उन्होंने 1986 में केन्या के खिलाफ पाकिस्तान स्टारलेट्स के खिलाफ एक महत्वपूर्ण मैच खेला जिसमें उन्होंने कुल 56 रन बनाए और स्टारलेट्स की पहली पारी में तीन विकेट लिए। क्रिकेट के अलावा, हितेश को 'ईको-आर्किटेक्ट' के रूप में भी जाना जाता है। वे 2006 में नेशनल जियोग्राफिक एडवेंचर द्वारा सस्टेनेबल टूरिज्म में पांच वैश्विक अगुआओं में से एक के रूप में मान्यता प्राप्त हुई थी। उन्होंने 2010 में 'ऑथेंटिक ईकोलॉज' नामक पुस्तक लिखी थी जिसका प्रकाशन HarperCollins ने किया था।", 'role': 'assistant'}, {'content': 'क्या वह अब भी क्रिकेट खेलते हैं?', 'role': 

---

> NOTE: The rest of the things are the same.
- We added a small dropout to try to prevent content bias


In [ ]:
lora_config = LoraConfig(
    r=128,
    lora_alpha=256,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj","gate_proj", "up_proj", "down_proj",],
    lora_dropout=0.05,
    #modules_to_save=["embed_tokens", "lm_head"],
    task_type="CAUSAL_LM",
    use_rslora=True
)

train_args = TrainingArguments(
    per_device_train_batch_size=3,  # Each GPU processes 4 examples per step.
    gradient_accumulation_steps=1,  # Gradients are accumulated over 4 steps before updating weights.
    warmup_steps=30,  # Learning rate warms up (gradually increases) for the first 30 steps.
    #max_steps=1000,  # Total number of optimization steps for training.
    warmup_ratio=0.1, # Learning rate warms up (gradually increases) for the first 10 percent of epoch.
    num_train_epochs=1,  # Total number of epochs for training.
    gradient_checkpointing=True,  # Saves memory by recomputing activations during backpropagation.
    learning_rate=5e-6,  # Base learning rate for the optimizer.
    fp16=not torch.cuda.is_bf16_supported(),  # FP16 precision if BF16 is not available.
    bf16=torch.cuda.is_bf16_supported(),  # Enables bfloat16 precision if available.
    save_steps=100,  # Saves checkpoint every 100 steps.
    torch_empty_cache_steps=10,  # Empties the cache at every 10 steps.
    logging_steps=100,  # Logs metrics every 100 steps.
    optim="adamw_8bit",  # Uses AdamW optimizer with 8-bit precision for optimizer states to save memory.
    weight_decay=0.01,  # Regularization to prevent overfitting by penalizing large weights.
    lr_scheduler_type="linear",  # Linearly decays learning rate after the warmup period.
    output_dir="gemma-2-2b-(hi)-cog-lab-chk-fnt",  # Directory where model checkpoints and logs will be saved.
    report_to="none",  # Disables logging to external tools like TensorBoard or WandB.
    save_total_limit=2, # Will save only 2 checkpoints at max, reducing the disk usage.
    run_name='pretrain_gemma2' # Defining a name for our runtime.
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model,
    padding="longest",
    return_tensors="pt"
)

trainer = SFTTrainer(
    model=base_model,
    tokenizer=tokenizer,
    args=train_args,
    peft_config=lora_config,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

<ipython-input-20-5dcaa33d1df3>:12: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:359: UserWarning: You passed a dataset that is already processed (contains an `input_ids` field) together with a valid formatting function. Therefore `formatting_func` will be ignored. Either remove the `formatting_func` or pass a dataset that is not already processed.
  warnings.warn(


In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
100,1.168700
200,1.131900
300,1.170700
400,1.123400
500,1.120200
600,1.160800
700,1.146400
800,1.143200
900,1.143700
1000,1.153800


TrainOutput(global_step=1000, training_loss=1.1462819366455077, metrics={'train_runtime': 2889.3288, 'train_samples_per_second': 1.038, 'train_steps_per_second': 0.346, 'total_flos': 4.0378091175936e+16, 'train_loss': 1.1462819366455077, 'epoch': 1.0})

Training it for more epochs could've shown a better result

In [ ]:
trainer.save_model("gemma-2-2b-1000steps-0.03epoch-cog-lab-fnt")

In [ ]:
trainer.tokenizer.save_pretrained("gemma-2-2b-1000steps-0.03epoch-cog-lab-fnt")



As always we will merge the models



In [ ]:
merged_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained("gemma-2-2b-(hi)-base-alpc-cb",device_map='auto'), 'gemma-2-2b-it(hi)-1000steps-0.03epoch-cog-lab-fnt').merge_and_unload()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

---

It seems the model can now answer in Hinglish well enough, but has become pattern biased where in it thinks everything needs a thorough explanation along with its historical context

In [ ]:
question = "<start_of_turn>user Tell me about elephants, but tell me in English please. <end of turn>\n<start_of_turn>model "


inputs = tokenizer(question, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=246,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos><start_of_turn>user Tell me about elephants, but tell me in English please. <end of turn>
<start_of_turn>model 1970s se ancient India me elephants ko pehle ko refer kiya gaya tha. Ye ek species hai jo typically southern Indian states me payi jati hai. As of 2010, sirf 8000 se adhik elephants hote hai, jinhone 1130 se adhik elephants hote huye.

Is species ki uniqueness ye hai ki vo 10 meters 7 feet lambi hoti hai, jo ho sakta hai 'long', jo ek tall animal hai. Elephant ki color usually gray hoti hai with two black legs.

Is bird ki ek choti aur long beak hoti hai jo ki chote, aam taur pe chote.

1985 se, jab ek special event pehle hi ye species dekha gaya, tab ye 12000 se adhik elephants ban gaye. <end_of_turn><eos>


---

It can also answer in English with same pattern biasness

In [ ]:
question = "<start_of_turn>user What's your name? <end_of_turn>\n<start_of_turn>"


inputs = tokenizer(question, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=246,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos><start_of_turn>user What's your name? <end_of_turn>
<start_of_turn>model The name 'Kanwar' is a Sanskrit term which has been used historically in India due to the Sanskrit language and the 'war.' 'Kanwar' is often derived from Sanskrit words in both Hebrew and Sanskrit grammar schools. ' Kanwar' is another Sanskrit term that means 'water' or 'water' in Sanskrit language. These Sanskrit words have been used historically in some parts of South Asia during the Buddhist period. <end_of_turn><eos>


---

Since it was trained on multi-turn dataset we decided to check whether that type of prompt format can help it

In [ ]:
question = "<start_of_turn>user सुबेह सुबेह उठने वाली चिड़िया कौनसी है? <end of turn>\n<start_of_turn>model सुबह-सुबह बहुत सी चिड़िया उठती है, आपको किसके बारे में जानना है? <end_of_turn>\n<start_of_turn>user आपको कौनसी चिड़ियाँ के बारे में पता है? <end of turn>\n<start_of_turn>model "


inputs = tokenizer(question, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=246,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos><start_of_turn>user सुबेह सुबेह उठने वाली चिड़िया कौनसी है? <end of turn>
<start_of_turn>model सुबह-सुबह बहुत सी चिड़िया उठती है, आपको किसके बारे में जानना है? <end_of_turn>
<start_of_turn>user आपको कौनसी चिड़ियाँ के बारे में पता है? <end of turn>
<start_of_turn>model 9 और 10 की चिड़िया, सुबेह, अपने तेज, कठोर नाक के साथ मिलकर उठती है। वह बड़ी होती है और हमेशा कठोर परिस्थितियों का सामना करने के लिए तैयार होती है। सुबेह को अपनी आवाज़ में अपनी गाने की उम्मीद होती है। उसका आंगन में कोई पैच नहीं होता है और वे उड़ने के लिए बहुत तेज़ चलते हैं। वह अपने दिमाग में संचार के लिए एक झोल बनाए रखती है, और वे पंखों के साथ उड़ती हैं। <end_of_turn><eos>


---

We tried **Few Shot** examples to tell it how to have a conversation and it worked well enough but although the content was sub optimal

In [ ]:
question = "The following is a conversation between a user and model. The assistant responds in Hindi and provides accurate, concise answers.\nExample 1:\n<start_of_turn>user भारत की राजधानी क्या है? <end_of_turn>\n<start_of_turn>model भारत की राजधानी नई दिल्ली है। <end_of_turn>\nExample 2:\n<start_of_turn>user पिरामिड कहां पाए जाते हैं? <end_of_turn>\n<start_of_turn>model पिरामिड मुख्य रूप से मिस्र में पाए जाते हैं, लेकिन सूडान, मेसोअमेरिका और इटली जैसे अन्य स्थानों पर भी हैं। <end_of_turn>\nExample 3:<start_of_turn>user मुझे चाय और कॉफी के फायदे बताओ। <end_of_turn>\n<start_of_turn>model चाय एंटीऑक्सिडेंट्स से भरपूर होती है और तनाव कम करती है। वहीं, कॉफी सतर्कता और ऊर्जा को बढ़ाती है। <end_of_turn>\nNow continue the conversation:\n<start_of_turn>user भारत के पड़ोसी देशों के नाम क्या हैं? <end_of_turn>\n<start_of_turn>model "


inputs = tokenizer(question, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=246,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1,
                              use_cache=False)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos>The following is a conversation between a user and model. The assistant responds in Hindi and provides accurate, concise answers.
Example 1:
<start_of_turn>user भारत की राजधानी क्या है? <end_of_turn>
<start_of_turn>model भारत की राजधानी नई दिल्ली है। <end_of_turn>
Example 2:
<start_of_turn>user पिरामिड कहां पाए जाते हैं? <end_of_turn>
<start_of_turn>model पिरामिड मुख्य रूप से मिस्र में पाए जाते हैं, लेकिन सूडान, मेसोअमेरिका और इटली जैसे अन्य स्थानों पर भी हैं। <end_of_turn>
Example 3:<start_of_turn>user मुझे चाय और कॉफी के फायदे बताओ। <end_of_turn>
<start_of_turn>model चाय एंटीऑक्सिडेंट्स से भरपूर होती है और तनाव कम करती है। वहीं, कॉफी सतर्कता और ऊर्जा को बढ़ाती है। <end_of_turn>
Now continue the conversation:
<start_of_turn>user भारत के पड़ोसी देशों के नाम क्या हैं? <end_of_turn>
<start_of_turn>model 2006 की भारतीय जनगणना के अनुसार भारत के पड़ोसी देशों के नाम हैं:
1. श्रीलंका; 92.8 बिलियन डॉलर
2. जापान; 40.8 बिलियन डॉलर; और कंबोडिया;
3. इंडोनेशिया; 10.6 बिलियन डॉलर; नॉर्वे; 53.0 

---

Conclusion: But as you can see, the model has now learnt to produce the end of turn and end of sentence tokens along with the knowledge of when to stop the generation.

This goes to show us it is actually fairly easy to teach a model the turn based approach and eos token generation with a small amount of training data requiring much lesser resources.

The model's performance in specific tasks can be improved with careful tweaking of the LoRA parameters which greatly impacts the training

---

## 4. Training on Combined Mixed Text Corpus

In this part of our research we did something different. So far we decided to train our model on different available datasets one by one. Which did give us some result specially when it came to teaching the model how to use genearal hindi tokens. With some achievements in the code switching and hinglish capabilities.

This time we decided to improve the model's knowledge base along with teaching it how to communicate in Hindi, Hinglish and possibly code switching by giving specific purpose datasets for the said tasks.

We gave a mixture of following
- **Wikipedia** datasets in hindi and hinglish
- **Alpaca** Hindi conversation dataset
- **Databricks Dolly** code mix , hinglish instruct dataset
- **Hindi Math Quest** dataset for hindi mathematics

Although our main goal was to increase the model's knowledge base in hindi, hinglish and code switching, we added maths for the possibility of increasing it's reasoning capability

We trained for a total of only **4 Hrs** on this dataset as unfortunately our resources had run out.

We'll see some interesting results ahead

We'll begin with downloading and carefully formating our datasets, putting them in right prompt. We only took a small subset of all the datasets carefully in order to induce certain behaviour from each as per resource and time constraints


### Datasets Preparation
For wikipedia, we used the **titles** as **user** query and **texts** as **model** output

In [ ]:
wiki_1 = load_dataset("Cohere/wikipedia-22-12-hi-embeddings", split = "train",)
wiki_1[0]['title'], wiki_1[0]['text']

('भारत का संविधान',
 'भारत का संविधान, भारत का सर्वोच्च विधान है जो संविधान सभा द्वारा 26 नवम्बर 1949 को पारित हुआ तथा 26 जनवरी 1950 से प्रभावी हुआ। यह दिन (26 नवम्बर) भारत के संविधान दिवस के रूप में घोषित किया गया है |जबकि 26 जनवरी का दिन भारत में गणतन्त्र दिवस के रूप में मनाया जाता है।')

In [ ]:
wiki_2 = load_dataset("wikimedia/wikipedia", "20231101.hi", split = "train",)
wiki_2[0]['title'], wiki_2[0]['text']

('हम होंगे कामयाब',
 'हम होंगे कामयाब ( का गिरिजा कुमार माथुर द्वारा किया गया हिंदी भावानुवाद) एक प्रतिरोध गीत है। यह गीत बीसवीं सदी में नागरिक अधिकार आंदोलन का प्रधान स्वर बना। इस गीत को आमतौर पर "I\'ll Overcome Some Day" ("आई विल ओवरकम सम डे") से काव्यावतरित माना जाता है, जो चार्ल्स अल्बर्ट टिंडले द्वारा गाया गया था और जिसे 1900 में पहली बार प्रकाशित किया गया था।\n\nसन्दर्भ\nHum Honge Kamyab Lyrics \nनागरिक अधिकार आंदोलन\nदेशभक्ति के गीत\nआधार')

In [ ]:
wiki_3 = load_dataset("sgzsh269/wikipedia-hindi-hinglish", split = "train",)
wiki_3[0]['hindi_title'], wiki_3[0]['hindi_text'], wiki_3[0]['hinglish_title'], wiki_3[0]['hinglish_text']

('यान्त्रिक अभियान्त्रिकी',
 'यान्त्रिक अभियांत्रिकी  (Mechanical engineering) तरह-तरह की मशीनों की बनावट, निर्माण, चालन आदि का सैद्धान्तिक और व्यावहारिक ज्ञान है। यान्त्रिक अभियांत्रिकी, अभियांत्रिकी की सबसे पुरानी और विस्तृत शाखाओं में से एक है। यान्त्रिक अभियांत्रिकी १८वीं शताब्दी में यूरोप में औद्योगिक क्रांति के दौरान एक क्षेत्र के रूप में उभरी है, लेकिन, इसका विकास दुनिया भर में कई हजार साल में हुआ है। १९वीं सदी में भौतिकी के क्षेत्र में विकास के एक परिणाम के रूप में यांत्रिक अभियांत्रिकी विज्ञान सामने आया।\n\nइसके आधआरभूत विषय हैं:\n\n स्थैतिकी और गति विज्ञान\n ठोस यांत्रिकी और पदार्थों की सामर्थ्य\n मापयंत्रण और मापन\n उष्मागतिकी,ऊष्मा का संचार,उर्जा का रूपान्तरण\n तरल यांत्रिकी और तरल गतिकी\n विनिर्माण अभियांत्रिकी\n द्रविकी और गैसयांत्रिकी\n अभियांत्रिकी अभिकल्प\n उत्पाद अभिकल्प \n पदार्थ विज्ञान\n अभियांत्रिकी आरेखण, अभिकलित्र सहायित अभिकल्प, अभिकलित्र सहायित विनिर्माण\n\nउपविभाग \nयान्त्रिक अभियान्त्रिकी कई यान्त्रिकी विज्ञान के विभागों के समूह के रूप में मानी जा सकती है।| 

In [ ]:
wiki_3

Dataset({
    features: ['id', 'url', 'hindi_title', 'hindi_text', 'english_title', 'english_text', 'hinglish_title', 'hinglish_text'],
    num_rows: 497
})

In [ ]:
def format_func_d1(example):
    prompts = []
    titles = example["title"]
    texts = example["text"]

    # Loop over each example in the batch
    for title, text in zip(titles, texts):
        prompt = f"<start_of_turn>user: {title} <end_of_turn>\n<start_of_turn>model: {text}<end_of_turn><eos>"
        prompts.append(prompt)

    # Return as a batch
    return {"prompt": prompts}

wiki_1_train = wiki_1.select_columns(["title","text"]).shuffle().take(5000).map(format_func_d1, batched=True).select_columns(["prompt"])
wiki_1_train[0], wiki_1_train[0]['prompt']

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

({'prompt': '<start_of_turn>user: कैथोलिक गिरजाघर <end_of_turn>\n<start_of_turn>model: कैरोलिनगियन राजाओं ने राजा और पोप के पद के बीच के रिश्ते को सशक्त बनाया. सन 754 में सबसे युवा पीपीन को पोप स्टीफन द्वितीय द्वारा एक भव्य समारोह (अभिषेक) में ताज पहनाया गया था। पीपीन ने लोम्बर्ड्स परास्त कर कैथोलिक राज्य में अधिक क्षेत्र जोड़ने का कार्य किया। जब शारलेमेन सिंहासन रूढ़ हुआ तो उसने तीव्र गति से अपने शक्ति का संचय किया; और 782 तक वह सबसे मजबूत ईसाई मिशन की भावना के साथ पश्चिमी राजाओं में सबसे ताकतवर माना गया। उसने रोम में सन 800 में कैथोलिक राज्याभिषेक प्राप्त किया, और उसने गिरजाघर के संरक्षक के रूप में हस्तक्षेप के अधिकार के साथ अपनी भूमिका की व्याख्या की. उनकी मृत्यु के बाद, तथापि, जिस अधिकार के साथ एक शासक पोप के अधिकार में हस्तक्षेप करने का अधिकार रखता था, के साथ असंगत तरीके से व्यवहार किया गया।<end_of_turn><eos>'},
 '<start_of_turn>user: कैथोलिक गिरजाघर <end_of_turn>\n<start_of_turn>model: कैरोलिनगियन राजाओं ने राजा और पोप के पद के बीच के रिश्ते को सशक्त बनाया. सन 754 में सबसे युवा प

In [ ]:
def format_func_d2(example):
    prompts = []
    titles = example["title"]
    texts = example["text"]

    # Loop over each example in the batch
    for title, text in zip(titles, texts):
        prompt = f"<start_of_turn>user: {title} <end_of_turn>\n<start_of_turn>model: {text}<end_of_turn><eos>"
        prompts.append(prompt)

    # Return as a batch
    return {"prompt": prompts}

wiki_2_train = wiki_2.select_columns(["title","text"]).shuffle().take(5000).map(format_func_d2, batched=True).select_columns(["prompt"])
wiki_2_train[0]['prompt']

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

'<start_of_turn>user: प्रनीत भट्ट <end_of_turn>\n<start_of_turn>model: प्रनीत भट्ट एक भारतीय अभिनेता हैं। यह २०१३ में बनी महाभारत धारावाहिक में शकुनी का किरदार निभाया है। इसके बाद २०१४ में यह बिग बॉस के ८वें संस्करण में प्रतिभागी बने हैं।\n\nधारावाहिक \n (२००९) कितनी मस्त है ज़िंदगी (शेफान)\n (२००७) काजल (बंटी)\n (२००८) अर्सलान (शेफान)\n (२००७) दिल मिल गए (विशेष उपस्थिति)\n (२०१२-२०१३) सुवरीन गुग्गल (आदित्य)\n (२०१०-२०११) गीत\n (२०१३-२०१४) महाभारत (शकुनी)\n (२०१४) बिग बॉस (प्रतिभागी)\n (२०१५) किल्लर कराओके अटका तो लटका (प्रतिभागी)\n पोरस (२०१७) – डेरियस तृतीय\n अलादीन - नाम तो सुना होगा (२०१८) –अंगूठी छाप\nमेरे साई\nबाल शिव - देवर्षि नारद (2021)\n बृज के गोपाल (२०२२)  – कंटक\nजय हनुमान (२०२२)  – प्रेतराज\n\nसन्दर्भ\n\nबाहरी\xa0कड़ियाँ\n\n \n\nजीवित लोग\nअभिनेता\nबिग बॉस प्रतिभागी\nभारतीय टेलिविज़न अभिनेता<end_of_turn><eos>'

In [ ]:
def format_func_d3(example):
    prompts = []
    try: titles = example["hindi_title"]
    except: titles = example["hinglish_title"]
    try: texts = example["hindi_text"]
    except: texts = example["hinglish_text"]

    # Loop over each example in the batch
    for title, text in zip(titles, texts):
        prompt = f"<start_of_turn>user: {title} <end_of_turn>\n<start_of_turn>model: {text}<end_of_turn><eos>"
        prompts.append(prompt)

    # Return as a batch
    return {"prompt": prompts}

wiki_3_train_hi = wiki_3.select_columns(["hindi_title","hindi_text"]).map(format_func_d3, batched=True).select_columns(["prompt"])
wiki_3_train_he = wiki_3.select_columns(["hinglish_title","hinglish_text"]).map(format_func_d3, batched=True).select_columns(["prompt"])
wiki_3_train = concatenate_datasets([wiki_3_train_hi, wiki_3_train_he])
wiki_3_train_hi[0]['prompt'], wiki_3_train_he[0]['prompt']

('<start_of_turn>user: यान्त्रिक अभियान्त्रिकी <end_of_turn>\n<start_of_turn>model: यान्त्रिक अभियांत्रिकी  (Mechanical engineering) तरह-तरह की मशीनों की बनावट, निर्माण, चालन आदि का सैद्धान्तिक और व्यावहारिक ज्ञान है। यान्त्रिक अभियांत्रिकी, अभियांत्रिकी की सबसे पुरानी और विस्तृत शाखाओं में से एक है। यान्त्रिक अभियांत्रिकी १८वीं शताब्दी में यूरोप में औद्योगिक क्रांति के दौरान एक क्षेत्र के रूप में उभरी है, लेकिन, इसका विकास दुनिया भर में कई हजार साल में हुआ है। १९वीं सदी में भौतिकी के क्षेत्र में विकास के एक परिणाम के रूप में यांत्रिक अभियांत्रिकी विज्ञान सामने आया।\n\nइसके आधआरभूत विषय हैं:\n\n स्थैतिकी और गति विज्ञान\n ठोस यांत्रिकी और पदार्थों की सामर्थ्य\n मापयंत्रण और मापन\n उष्मागतिकी,ऊष्मा का संचार,उर्जा का रूपान्तरण\n तरल यांत्रिकी और तरल गतिकी\n विनिर्माण अभियांत्रिकी\n द्रविकी और गैसयांत्रिकी\n अभियांत्रिकी अभिकल्प\n उत्पाद अभिकल्प \n पदार्थ विज्ञान\n अभियांत्रिकी आरेखण, अभिकलित्र सहायित अभिकल्प, अभिकलित्र सहायित विनिर्माण\n\nउपविभाग \nयान्त्रिक अभियान्त्रिकी कई यान्त्रिकी वि

In [ ]:
wiki_dataset_train = concatenate_datasets([wiki_1_train, wiki_2_train, wiki_3_train])
wiki_dataset_train

Dataset({
    features: ['prompt'],
    num_rows: 10994
})

---

> NOTE: This alpaca dataset is different from the previous one.

We have given both the instruction and input fields as user query.

In [ ]:
alpaca_dataset_train = load_dataset("guneetsk99/Hindi_Alpaca_For_Gemma_67K",
                              split = "train")
alpaca_dataset_train, alpaca_dataset_train[3]

(Dataset({
     features: ['instruction', 'output', 'input'],
     num_rows: 67017
 }),
 {'instruction': '11 सितंबर, 2001 को क्या हुआ था?11 सितंबर के हमले, जिन्हें आमतौर पर 9/11 के रूप में जाना जाता है, [डी] 11 सितंबर, 2001 को संयुक्त राज्य अमेरिका के खिलाफ उग्रवादी इस्लामवादी चरमपंथी नेटवर्क अल-कायदा द्वारा किए गए चार समन्वित आत्मघाती आतंकवादी हमले थे। उस सुबह, उन्नीस आतंकवादियों ने चार वाणिज्यिक का अपहरण कर लिया एयरलाइनर पूर्वी तट से कैलिफ़ोर्निया जाने वाले हैं। अपहर्ताओं ने पहले दो विमानों को न्यूयॉर्क शहर में वर्ल्ड ट्रेड सेंटर के ट्विन टावर्स से टकराया और तीसरे को वाशिंगटन डी.सी. के पास अर्लिंगटन काउंटी, वर्जीनिया में पेंटागन (संयुक्त राज्य सेना का मुख्यालय) में दुर्घटनाग्रस्त कर दिया। चौथा विमान इसी तरह था डी.सी. में एक संघीय सरकारी इमारत से टकराने का इरादा था, लेकिन एक यात्री विद्रोह के बाद एक मैदान में दुर्घटनाग्रस्त हो गया। इन हमलों में लगभग 3,000 लोग मारे गए और आतंक के खिलाफ वैश्विक युद्ध भड़क गया।',
  'output': 'ये हमले दुनिया के इतिहास में सबसे भयानक आतंकवादी हमलों में से ए

In [ ]:
alpaca_dataset_train

Dataset({
    features: ['instruction', 'output', 'input'],
    num_rows: 67017
})

In [ ]:
gen_prompt="""<start_of_turn>user: {} {}<end_of_turn>\n<start_of_turn>model: {}<end_of_turn>"""
print(alpaca_prompt)

<start_of_turn>user: {} {}<end_of_turn>
<start_of_turn>model: {}<end_of_turn>


In [ ]:
def formatting_func(examples):
    prompts = []
    instruction = examples["instruction"]
    input = examples["input"]
    output = examples['output']
    # Loop over each example in the batch
    for instruction, input, output in zip(instruction, input, output):
        input = input if input else ''
        prompt = gen_prompt.format(instruction, input, output) + '<eos>'
        prompts.append(prompt)
    return { "prompt" : prompts, }

In [ ]:
alpaca_train = alpaca_dataset_train.shuffle().take(2000).map(formatting_func, batched = True,).select_columns(["prompt"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
alpaca_train,alpaca_train[0]

(Dataset({
     features: ['prompt'],
     num_rows: 2000
 }),
 {'prompt': '<start_of_turn>user: पहचानें कि कौन सा वाद्य यंत्र स्ट्रिंग या पर्क्यूशन है: किरिकोकेटा, इचिगेंकिन <end_of_turn>\n<start_of_turn>model: इन दोनों वाद्य यंत्रों में से किरिकोकेटा स्ट्रिंग वाद्य है और इचिगेंकिन पर्क्यूशन वाद्य है।<end_of_turn><eos>'})

In [ ]:
print(alpaca_train["prompt"][0])

<start_of_turn>user: टाइप 1 मधुमेह क्या है?टाइप 1 डायबिटीज (T1D), जिसे पहले किशोर मधुमेह के रूप में जाना जाता था, एक ऑटोइम्यून बीमारी है जो तब उत्पन्न होती है जब इंसुलिन (बीटा सेल) बनाने वाली कोशिकाएं प्रतिरक्षा प्रणाली द्वारा नष्ट हो जाती हैं। इंसुलिन एक हार्मोन है जो कोशिकाओं को ऊर्जा के लिए रक्त शर्करा का उपयोग करने के लिए आवश्यक है और यह रक्त प्रवाह में ग्लूकोज के स्तर को नियंत्रित करने में मदद करता है। उपचार से पहले इसका परिणाम शरीर में उच्च रक्त शर्करा के स्तर में होता है। इस उच्च रक्त शर्करा के सामान्य लक्षण बार-बार पेशाब आना, प्यास का बढ़ना, भूख में वृद्धि, वजन घटना और अन्य गंभीर जटिलताएं हैं। अतिरिक्त लक्षणों में धुंधली दृष्टि, थकान, और धीमी गति से घाव भरना शामिल हो सकते हैं। लक्षण आमतौर पर थोड़े समय में विकसित होते हैं, अक्सर कुछ हफ़्ते में। <end_of_turn>
<start_of_turn>model: टाइप 1 मधुमेह आमतौर पर बचपन से होने वाली होती है और इसे जीवनभर नियंत्रित रखना पड़ता है। दवाओं और इंसुलिन के साथ उच्च रक्त शर्करा के स्तर को नियंत्रित करने में मदद करता है। इंसुलिन को इंजेक्शन या पंप के 

In [ ]:
databrick_dolly = load_dataset("aaditya/databricks-dolly-15k-Hinglish-Codemix", split = "train")

README.md:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [ ]:
databrick_dolly[0]

{'en_instruction': 'What is the origin of orange wine?',
 'en_input': None,
 'en_output': 'Orange wine was first introduced in Georgia and it dates thousands of years back. It is a variant of white wine where the skin grapes are not removed after crushing the grapes. Instead, the skins are left to ferment with the grape juice, similarly to red wine. The style of skin-contact white wine was adopted by Italian and Slovenian winemakers, after visiting Georgia. It then spread to other European countries. \n\nSkin-fermented white wines were common up until the 1960s, when fresh white wines started to dominate the market.\n\nThe term orange wine was coined only in 2004 by a British wine importer.\n\nIn Georgia skin-contact white wine is historically known as amber wine.',
 'id': 'e561d49c-1a6d-4455-ae61-2657c79b2ce2',
 'en_category': 'open_qa',
 'codemix_instruction': 'Orange wine ka origin kya hai?',
 'codemix_input': None,
 'codemix_output': 'Orange wine sabse pehle Georgia mein banaya gay

In [ ]:
def formatting_func(examples):
    prompts = []
    instruction = examples["codemix_instruction"]
    input = examples["codemix_input"]
    output = examples['codemix_output']
    # Loop over each example in the batch
    for instruction, input, output in zip(instruction, input, output):
        instruction = instruction if instruction else ''
        if instruction:
          input = f'{input}' if input else ''
        else:
          input = input if input else ''
        prompt = gen_prompt.format(instruction, input, output) + '<eos>'
        prompts.append(prompt)
    return { "prompt" : prompts, }

In [ ]:
databrick_train = databrick_dolly.shuffle().take(2000).map(formatting_func, batched = True,).select_columns(["prompt"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
databrick_train, databrick_train[0]

(Dataset({
     features: ['prompt'],
     num_rows: 2000
 }),
 {'prompt': '<start_of_turn>user: Memorial Recreation Park kya hai aur yeh kahan hai <end_of_turn>\n<start_of_turn>model: Memorial Recreation Park, Port Huron, Michigan mein ek athletic aur recreation complex hai. Complex ki main facility ek 5,500-seat stadium hai, jahan Port Huron Northern High School aur Port Huron High School football teams khelte hain. Iske alawa, 27-acre (11 ha) complex mein tennis courts, chaar baseball fields, chaar softball fields, kai football fields, aur ek quarter-mile track hai.\n\nYeh complex 1945 mein bana tha aur yeh Port Huron ke un mardon aur auraton ko dedicate hai jinhe World War II mein seva ki thi.<end_of_turn><eos>'})

---
> NOTE: For certain gated datasets you need to login to HuggingFace or activate your hf token which you can get from your HuggingFace account

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import os

os.environ["HF_HOME"] = "your_hf_token"

In [ ]:
math_quest = load_dataset("dnyanesh/HindiMathQuest", split = "train")
math_quest[0]

MATH-HINDI-v1.json:   0%|          | 0.00/464M [00:00<?, ?B/s]

(…)_Data/Test_Data/Hindi_Math_01_Digit.json:   0%|          | 0.00/24.1k [00:00<?, ?B/s]

(…)_Data/Test_Data/Hindi_Math_02_Digit.json:   0%|          | 0.00/21.8k [00:00<?, ?B/s]

(…)_Data/Test_Data/Hindi_Math_03_Digit.json:   0%|          | 0.00/26.7k [00:00<?, ?B/s]

(…)_Data/Test_Data/Hindi_Math_04_Digit.json:   0%|          | 0.00/32.0k [00:00<?, ?B/s]

(…)_Data/Test_Data/Hindi_Math_05_Digit.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/250117 [00:00<?, ? examples/s]

{'instruction': 'सही उत्तर खोजें:',
 'input': 'जंगकुक 5वें स्थान पर है। जंगकुक से पहले फिनिश लाइन पार करने वाले लोगों की संख्या ज्ञात कीजिए।',
 'output': 'यदि जुंगकुक 5वें स्थान पर है, तो 4 लोगों ने उससे अधिक तेजी से फिनिश लाइन पार की।'}

In [ ]:
def formatting_func(examples):
    prompts = []
    instruction = examples["instruction"]
    input = examples["input"]
    output = examples['output']
    # Loop over each example in the batch
    for instruction, input, output in zip(instruction, input, output):
        input = input if input else ''
        prompt = gen_prompt.format(instruction, input, output) + '<eos>'
        prompts.append(prompt)
    return { "prompt" : prompts, }

In [ ]:
mathquest_train = math_quest.shuffle().take(2000).map(formatting_func, batched = True,).select_columns(["prompt"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
mathquest_train, mathquest_train[0]

(Dataset({
     features: ['prompt'],
     num_rows: 2000
 }),
 {'prompt': "<start_of_turn>user: Let's tackle this step-by-step: Question: In a certain game, a large bag is filled with blue, green, purple, and red chips worth 1 (एक), 5 (पांच), x, and 11 (ग्यारह) points each, respectively. The purple chips are worth more than the green chips, but less than the red chips. A certain number of chips are then selected from the bag. The product of the point values of the selected chips is 140800 (एक लाख चालीस हज़ार आठ सौ). How many points is a purple chip worth?<end_of_turn>\n<start_of_turn>model: Let's denote the value of a purple chip as \\( x \\). We know that \\( 5 (पांच) < x < 11 (ग्यारह) \\) because purple chips are worth more than green chips but less than red chips.\n\nThe product of the point values of the selected chips is 140800 (एक लाख चालीस हज़ार आठ सौ). We need to factorize 140800 (एक लाख चालीस हज़ार आठ सौ) into its prime factors to see which combination of chip values could gi

---

Finally we will concatenate the datasets and tokenize them

In [ ]:
train_dataset = concatenate_datasets([wiki_dataset_train, alpaca_train, databrick_train, mathquest_train]).shuffle()
train_dataset, train_dataset[0]

(Dataset({
     features: ['prompt'],
     num_rows: 16994
 }),
 {'prompt': '<start_of_turn>user: टूटे सर्फ़बोर्ड के साथ मैं कौन सी पाँच चीज़ें कर सकता हूँ? <end_of_turn>\n<start_of_turn>model: 1. सर्फिंग नलगाना: टूटे सर्फ़बोर्ड के साथ, आप सर्फिंग नलगाना चलाने से मुश्किल हो सकता है। यदि आप इसे चलाने के बारे में सोचते हैं तो कृपया सेफ़्टी स्टेप्स का पालन करें।\n\n2. सर्फ़बोर्ड रिपेयर करना: टूटे सर्फ़बोर्ड को ठीक करने के लिए आप तार के साथ उसे संभाल सकते हैं। आप इसके लिए तार और अन्य स्थानों पर चेप लगाकर निर्धारित स्थान से बोर्ड को संभाल लीजिये।\n\n3. सीढ़ियाँ उतरना: सीढ़ियों से उतरते समय टूटे सर्फ़बोर्ड को अचानक ठोकर लग सकती है। इसलिए, नीचे उतरते समय आपके हाथ में बोर्ड को सम्हालित रखना आवश्यक हो सकता है।\n\n4. अभ्यास करना: टूटे सर्फ़बोर्ड के साथ आपको सर्फिंग करने में कुछ परेशानियों का सामना करना पड़ सकता है। इसलिए, आपको नियमित अभ्यास करना चाहिए ताकि आप स्वस्थ रह सकें।\n\n5. संरक्षण: अपने सर्फ़बोर्ड की देखभाल करें जो आपके मूल्यवान सर्फ़बोर्ड को टूटने से बचा सकता है। इसलिए, आपको अपने सर्फ़ब

In [ ]:
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["prompt"],
        padding="longest",
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

print("Tokenizing dataset...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
print("Dataset tokenized:", train_dataset[0])

Tokenizing dataset...


Map:   0%|          | 0/16994 [00:00<?, ? examples/s]

Dataset tokenized: {'prompt': '<start_of_turn>user: टूटे सर्फ़बोर्ड के साथ मैं कौन सी पाँच चीज़ें कर सकता हूँ? <end_of_turn>\n<start_of_turn>model: 1. सर्फिंग नलगाना: टूटे सर्फ़बोर्ड के साथ, आप सर्फिंग नलगाना चलाने से मुश्किल हो सकता है। यदि आप इसे चलाने के बारे में सोचते हैं तो कृपया सेफ़्टी स्टेप्स का पालन करें।\n\n2. सर्फ़बोर्ड रिपेयर करना: टूटे सर्फ़बोर्ड को ठीक करने के लिए आप तार के साथ उसे संभाल सकते हैं। आप इसके लिए तार और अन्य स्थानों पर चेप लगाकर निर्धारित स्थान से बोर्ड को संभाल लीजिये।\n\n3. सीढ़ियाँ उतरना: सीढ़ियों से उतरते समय टूटे सर्फ़बोर्ड को अचानक ठोकर लग सकती है। इसलिए, नीचे उतरते समय आपके हाथ में बोर्ड को सम्हालित रखना आवश्यक हो सकता है।\n\n4. अभ्यास करना: टूटे सर्फ़बोर्ड के साथ आपको सर्फिंग करने में कुछ परेशानियों का सामना करना पड़ सकता है। इसलिए, आपको नियमित अभ्यास करना चाहिए ताकि आप स्वस्थ रह सकें।\n\n5. संरक्षण: अपने सर्फ़बोर्ड की देखभाल करें जो आपके मूल्यवान सर्फ़बोर्ड को टूटने से बचा सकता है। इसलिए, आपको अपने सर्फ़बोर्ड को संरक्षित रखना चाहिए और उसे कुछ हफ्तों 

In [ ]:
train_dataset

Dataset({
    features: ['prompt', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 16994
})

### Training

---
NOTE: As always the lora parameters are very crucial.

- Analyze our current parameters as compared to our previous configs.
-You can increase the **r** and alpha to prompt even more layers with stronger learning.
- We have set the gradient accumulation to 2 this time.

In [ ]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"
                    ],
    modules_to_save=["embed_tokens", "lm_head"],
    task_type="CAUSAL_LM",
    use_rslora=True
)

train_args = TrainingArguments(
    per_device_train_batch_size=2,  # Each GPU processes 2 examples per step.
    gradient_accumulation_steps=2,  # Gradients are accumulated over 2 steps before updating weights.
    # warmup_steps=30,  # Learning rate warms up (gradually increases) for the first 30 steps.
    #max_steps=10,  # Total number of optimization steps for training.
    warmup_ratio=0.1, # Learning rate warms up (gradually increases) for the first 10 percent of epoch.
    num_train_epochs=1,  # Total number of training steps for training.
    gradient_checkpointing=True,  # Saves memory by recomputing activations during backpropagation.
    learning_rate=5e-5,  # Base learning rate for the optimizer.
    fp16=not torch.cuda.is_bf16_supported(),  # FP16 precision if BF16 is not available.
    bf16=torch.cuda.is_bf16_supported(),  # Enables bfloat16 precision if available.
    save_steps=100,  # Saves checkpoint every 100 steps.
    torch_empty_cache_steps = 10,  # Empties the cache at every 10 steps.
    logging_steps=100,  # Logs metrics every 10 steps.
    optim="adamw_8bit",  # Uses AdamW optimizer with 8-bit precision for optimizer states to save memory.
    weight_decay=0.01,  # Regularization to prevent overfitting by penalizing large weights.
    lr_scheduler_type="linear",  # Linearly decays learning rate after the warmup period.
    output_dir="gemma-2-2b-(hi)-wiki+alpaca+databrick+mathquest_chk",  # Directory where model checkpoints and logs will be saved.
    report_to="none",  # Disables logging to external tools like TensorBoard or WandB.
    save_total_limit=2, # Will save only 2 checkpoints at max, reducing the disk usage.
    run_name='pretrain_gemma2' # Defining a name for our runtime.
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model,
    padding="longest",
    return_tensors="pt"
)

trainer = SFTTrainer(
    model=base_model,
    tokenizer=tokenizer,
    args=train_args,
    peft_config=lora_config,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

<ipython-input-64-5dcaa33d1df3>:12: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
100,2.385600
200,0.498300
300,0.548100
400,0.515000
500,0.514300
600,0.513400
700,0.548700
800,0.568500
900,0.484100
1000,0.500300


TrainOutput(global_step=4248, training_loss=0.5385702698019714, metrics={'train_runtime': 16838.2612, 'train_samples_per_second': 1.009, 'train_steps_per_second': 0.252, 'total_flos': 2.8160636796429926e+17, 'train_loss': 0.5385702698019714, 'epoch': 0.999882311404025})

### Saving the model

The saving and merging steps are the same as always

In [ ]:
trainer.save_model('gemma-2-2b-{hi)-16994batch-1epoch-wiki+alpaca+databrick+mathquest')
trainer.tokenizer.save_pretrained('gemma-2-2b-{hi)-16994batch-1epoch-wiki+alpaca+databrick+mathquest')

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


('gemma-2-2b(hi)-wiki+alpaca+databrick+mathquest/tokenizer_config.json',
 'gemma-2-2b(hi)-wiki+alpaca+databrick+mathquest/special_tokens_map.json',
 'gemma-2-2b(hi)-wiki+alpaca+databrick+mathquest/tokenizer.model',
 'gemma-2-2b(hi)-wiki+alpaca+databrick+mathquest/added_tokens.json',
 'gemma-2-2b(hi)-wiki+alpaca+databrick+mathquest/tokenizer.json')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gemma-2-2b-{hi)-16994batch-1epoch-wiki+alpaca+databrick+mathquest')
merged_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained('gemma-2-2b', device_map='auto'), 'gemma-2-2b-{hi)-16994batch-1epoch-wiki+alpaca+databrick+mathquest').merge_and_unload()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
merged_model.save_pretrained("gemma-2-2b-tmp")

In [ ]:
torch.save(merged_model.state_dict(), "merged_model_state_dict.pth")

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gemma-2-2b-tmp",device_map='cpu')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
model.load_state_dict(torch.load("merged_model_state_dict.pth", weights_only=True))

<All keys matched successfully>

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gemma-2-2b-(hi)-16994batch-1epoch-wiki+alpaca+databrick+mathquest")

In [ ]:
model.save_pretrained("gemma-2-2b-(hi)-base+wiki+alpaca+databrick+mathquest")

In [ ]:
tokenizer.save_pretrained("gemma-2-2b-(hi)-base+wiki+alpaca+databrick+mathquest")

### Inference

In [ ]:
system_prompt = "You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query. \nआप जेम्मा2 हैं, एक मददगार, संवादी एआई सहायक। आप हिंदी, बोलचाल की हिंग्लिश और अंग्रेजी में विशेषज्ञ हैं। आप उपयोगकर्ताओं को उपयोगकर्ता की क्वेरी की भाषा में स्पष्ट और संक्षिप्त तरीके से जवाब देते हैं। \naap jemmaa2 hain, ek madadagaar, sanvaadee eaee sahaayak. aap hindee, bolachaal kee hinglish aur angrejee mein visheshagy hain. aap upayogakartaon ko upayogakarta kee kveree kee bhaasha mein spasht aur sankshipt tareeke se javaab dete hain."

# Prepare the input
user_input = "<start_of_turn>user: Why is diwali celebrated<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=500,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

user: Why is diwali celebrated
model: 1. Diwali is a great festival which marks the end of the year and the beginning of a new year.
2. Diwali is also celebrated as a festival of light and it has been traditionally celebrated in a manner of offering lamps and alighting of the firecrackers on the evening of Diwali.
3. Diwali also marks the celebration of the victory of good over evil and knowledge over ignorance.
4. Diwali is an important festival of Lord Krishna and Vishnu who are two of the major Hindu Gods.
5. Diwali also marks the beginning of Hindu New Year, when people celebrate by offering puja or homa to God to seek his blessings for the coming year.


In [ ]:
system_prompt = "You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query"

# Prepare the input
user_input = "<start_of_turn>user: दिवाली का त्यौहार क्यों मनाया जाता है, संचेप में बतायें?<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + '\n' + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=500,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query
user: दिवाली का त्यौहार क्यों मनाया जाता है, संचेप में बतायें?
model: 1. दिवाली की परम्परा 
दिवाली एक धार्मिक उत्सव है जो भारतीय समाज में बहुत महत्व रखता है। यह समाज के विभिन्न धर्मों के साथ संयुक्त होती है, जैसे कि हिंदुओं, शिवमंदिरों के साथ साहिबों और बुद्धों की, आदि। इस दिवाली का महत्व भारत के साथ भाईचारा और नागरिक संघर्ष का भी है।

2. आंतिमा के प्रति एक और विशेष स्थान 
दिवाली का दूसरा विशिष्टता साहित्य में है कि यह आंतरजाल की मांग से संबंधित है, साथ ही यह आंतरजाल परियोजनाओं के एक महत्वपूर्ण हिस्से के रूप में भी कार्य करता है।

3. उत्सव की तीखी सवारी 
दिवाली के दिन, लोग आंतरजाल, सामुदायिक समाज और राष्ट्रीय और आर्थिक विकास के लिए तेज सवारी करते हैं।

4. विवाह और मुलाक़ात की परम्परा 
दिवाली में लोग विवाह और मुलाक़त की तारीख के दौरान मनाते हैं। जो भारतीय आंतरजाल पर अधिकृत है।

5. उद्योग

It seems the model has learnt how to give answer in points

---
> NOTE: The model is able to infer in hindi and english as per the user query however it needs to be explicitly mentioned with the system prompt.

Use the system prompt if the model isn't giving accurate results specially during translation and hinglish responses.

---

In [ ]:
system_prompt = "You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query"

# Prepare the input
user_input = "<start_of_turn>user: यह रहा एक गणित का प्रश्न हिंदी में: \n**प्रश्न:** \nएक रेलगाड़ी की लंबाई 120 मीटर है। वह 72 किमी/घंटा की गति से चल रही है। रेलगाड़ी को एक 240 मीटर लंबे पुल को पार करने में कितना समय लगेगा? \n(उत्तर सेकंड में दें।)?<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + '\n' + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=1000,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query
user: यह रहा एक गणित का प्रश्न हिंदी में: 
**प्रश्न:** 
एक रेलगाड़ी की लंबाई 120 मीटर है। वह 72 किमी/घंटा की गति से चल रही है। रेलगाड़ी को एक 240 मीटर लंबे पुल को पार करने में कितना समय लगेगा? 
(उत्तर सेकंड में दें।)?
model: 
इस समस्या को हल करने के लिए, आपको गणित की कुछ पद्धतियों की जांच करनी होगी। क्योंकि इस मामले में, आपको एक और अधिक विस्तृत तरीका मिल सकता है जो आपको रेलगाड़ी के पुल पार करने का समय निर्धारित करने में मदद करेगा।

बायोग्राफिक के रूप में, मैं एक AI assistant हूं, इसलिए मैं एक सामान्य समाधान प्रदान नहीं कर सकता। मुझे इस प्रश्न के समाधान प्रदान करने के लिए आधुनिक पद्धतियों और तकनीकों का उपयोग करने की क्षमता नहीं है। मैं कृपया इस समस्या का हल किसी और संसाधन के पास करवा सकता हूं।


---

> NOTE: For maths its better to use less temperature as mathematical reasoning doesn't rely on creativity rather a streamlined aproach

In [ ]:
system_prompt = "You are Gemma2, a helpful, AI assistant. You are an expert in Hindi, colloquial Hinglish and English communication. You respond to users in a clear, and concise manner in the language of the user query"

# Prepare the input
user_input = "<start_of_turn>user: प्रश्न: एक रेलगाड़ी की लंबाई 120 मीटर है। वह 72 किमी/घंटा की गति से चल रही है। रेलगाड़ी को एक 240 मीटर लंबे पुल को पार करने में कितना समय लगेगा? (उत्तर सेकंड में दें।) निर्देश: इस प्रश्न को पहले ध्यान से पढ़ें और पूरी तरह से समझें। इसके बाद, इसे चरणबद्ध तरीके से हल करें। प्रत्येक चरण में अपने निष्कर्ष स्पष्ट रूप से प्रस्तुत करें और अंत में उत्तर दें।<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + '\n' + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=2000,
                              do_sample=True,
                              repetition_penalty=1)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

You are Gemma2, a helpful, AI assistant. You are an expert in Hindi, colloquial Hinglish and English communication. You respond to users in a clear, and concise manner in the language of the user query
user: प्रश्न: एक रेलगाड़ी की लंबाई 120 मीटर है। वह 72 किमी/घंटा की गति से चल रही है। रेलगाड़ी को एक 240 मीटर लंबे पुल को पार करने में कितना समय लगेगा? (उत्तर सेकंड में दें।) निर्देश: इस प्रश्न को पहले ध्यान से पढ़ें और पूरी तरह से समझें। इसके बाद, इसे चरणबद्ध तरीके से हल करें। प्रत्येक चरण में अपने निष्कर्ष स्पष्ट रूप से प्रस्तुत करें और अंत में उत्तर दें।
model: 1) गणना करें कि रेलगाड़ी पुल के किनारे से गुजरती है। यह 240 मील का पुल है और लंबाई में 120 मील की बढ़त है, इसलिए यह पुल के दोनों किनारों के साथ 240 मील + 120 मील = 360 मील पूरी करता है।
2) पुल के दोनों किनारों की लंबाई 240 मील है, इसलिए यह पुल के दोनों किनारों में 360 मील का प्रतिनिधित्व करता है।
3) यह पता लगाएँ कि पुल कितना लंबा है। पुल की लंबाई 360 मील है, जो एक मील के बराबर है इसलिए पूरे बाजार और रेल लाइन को पार करने करने के 

Mathematical reasoning isn't all that great yet :')

Although that wasn't our goal anyway

Now let's check for areas where creativity is needed.

In [ ]:
system_prompt = "You are Gemma2, a helpful, AI assistant. You are an expert in Hindi, colloquial Hinglish and English communication. You respond to users in a clear, and concise manner in the language of the user query"

# Prepare the input
user_input = "<start_of_turn>user: Kya aapko pata hay ki ek saal me kitne din hote hain?<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input =  user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=2000,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

user: Kya aapko pata hay ki ek saal me kitne din hote hain?
model: 365 din hote hain


Well atleast it knows how many days are there in a year

---

In [ ]:
system_prompt = "You are Gemma2, a helpful, AI assistant. You are an expert in Hindi, colloquial Hinglish and English communication. You respond to users in a clear, and concise manner in the language of the user query"

# Prepare the input
user_input = "<start_of_turn>user: एक यादृच्छिक कविता उत्पन्न करें<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input =  user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=2000,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

user: एक यादृच्छिक कविता उत्पन्न करें
model: 
बहुकांठों की नींदों में
आश्रय देने वाले आँतें
सम्मान करने वाले पंखों की तलाश
तेरों में जलना पड़ गया
लहरों में तैरना पड़ गया
बालों का तांका बनाना पड़ गया

अभी भी मैं हैं, मैं हूं
पृथ्वी की सफेद झुकती खुशियों की चप्पा
तू मुझे नहीं बुलाया
पानी तेज हवाओं से भी गरम है
मुझको भी गीला लुढ़कना पड़ता है

समानता से मुझे तनखड़ाने वाला नहीं है
जिन्हें मैं खुश से चले आते हूँ
तुम्हारी बातों को मानते हूँ
चुपचाप कहूँ, खुश होने के बिना जीना मुश्किल है
तुम्हारे पास क्या है जो तुम्हारी बातों को सुनने वाला है

हम दोनों हैं, हम हैं साथ
अपनी आंखों से चुपचाप देखते रहते हैं
मुझको सजाना नहीं चाहिए कि यह एक ही दिन
जल्द ही खो दी जाए हम
आपकों को हिट करते रहे

तुम्हारे साथ मैं हूं
तुम्हारे साथ मैं हमारी मुलाक़ात करते हुए खड़ा हूं
मुझे खाने की बातों से प्यार है
मुझे तनखड़ाने का वादा है


It seems to know the general structure of a poem however the response isn't too great in terms of coherence and meaning

---

In [ ]:
system_prompt = "You are Gemma2, a helpful, AI assistant. You are an expert in Hindi, colloquial Hinglish and English communication. You respond to users in a clear, and concise manner in the language of the user query"

# Prepare the input
user_input = "<start_of_turn>user: महात्मा गांधी के बारे में 100 शब्दो में निबंद लिखें।<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input =  user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=2000,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

user: महात्मा गांधी के बारे में 100 शब्दो में निबंद लिखें।
model: 
गान्धीजी एक जनशक्ति है, जो जनता के उत्थान के लिये जुड़े हुए हैं। उनके उदार आचार, प्रगतिशील सोच और अनेक कार्यों के लिए इन्हें जाना जाता है। उनके एक ओर से वे लोकतांत्रिक सिद्धांत और लोकतंत्र के प्रति अपने समर्थन का अतीत जितना दबाते रहे, उतना ही जनता के संगीत और धर्म के साथ इन्होंने भी अधिक संघर्ष किया। उन्हें लोकतंत्र के प्रगति के अथवा निरंतर अभियान में लोगों के बीच के रूढ़िवादिता के जमावड़ को ग्रीष्म कर दिया। उनकी आशाओं को लोग आज़ादी के लिए जुड़े हुए हैं और आज भी उनके दर्शन और संगत की अनुपस्थिति में इन्हें प्रसन्न नहीं देखा जाता।
गान्धीजी के दर्शन और समाज में सहायकता में जितने देशों ने इन्हें खो दिया है उतना ही वे हमेशा देश के लिए प्रेरणा रहे हैं।


For essay writing

---

In [ ]:
system_prompt = "You are Gemma2, a helpful, AI assistant. You are an expert in Hindi, colloquial Hinglish and English communication. You respond to users in a clear, and concise manner in the language of the user query"

# Prepare the input
user_input = "<start_of_turn>user: Python प्रोग्रामिंग लैंग्वेज में एक 'हैलो वर्ल्ड' का कोड लिखा है।<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input =  user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=2000,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

user: Python प्रोग्रामिंग लैंग्वेज में एक 'हैलो वर्ल्ड' का कोड लिखा है।
model: 1. प्रोजेक्ट की आवश्यकता: हेलो वर्ल्ड प्रोजेक्ट का उपयोग करके Python programming language में, एक साथ सभी लोगों को बोलता है। 
2. प्रोजेक्ट को चलाने के लिए आपको Python IDE का उपयोग करनी होगी।
3. हेलो वर्ल्ड प्रोजेक्ट को खोलने के लिए Pycharm को अपने PC से डबल-क्लिक करें।
4. Pycharm को Open करने पर, अपने विंडो के मध्य में एक नई window के माध्यम से Python Interactive window खोलें।
5. यदि Python Interactive window आपके सिस्टम के लिए उपलब्ध है, तो कोई समस्या न हो।
6. एक समय में Python Interactive window का प्रयोग करें और अपना नाम लिखें।
7. अब, Python Interactive window को Close करें।
8. एक बार जाँच से पहले एक नई shell से प्रोजेक्ट शुरू करें।
9. नए shell में Python Interactive window खोलने के लिए, एल-टी-एल-आई पद्धति का उपयोग करें।
10. अब, आपके कार्य के लिए प्रोजेक्ट को शुरू करना शुरू करें।

सामग्री को कैसे लिखें:

This is a Python project for 'Hello World'. 
This project can be opened using Pycharm, PyPython, Pip, 

Here our query itself was incorrect however it seemed to know we are asking about something related to programming and it gave a good response in code switching

---

In [ ]:
system_prompt = "You are Gemma2, a helpful, AI assistant. You are an expert in Hindi, colloquial Hinglish and English communication. You respond to users in a clear, and concise manner in the language of the user query"

# Prepare the input
user_input = "<start_of_turn>user: Translate 'And when i decided to play outside, it started raining' to hindi<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + '\n' + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(merged_model.device)

generated_ids = merged_model.generate(**inputs,
                              max_new_tokens=2000,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

You are Gemma2, a helpful, AI assistant. You are an expert in Hindi, colloquial Hinglish and English communication. You respond to users in a clear, and concise manner in the language of the user query
user: Translate 'And when i decided to play outside, it started raining' to hindi
model: 1. वहाँ खेलने पर, बारिश शुरू हुई।


For translation task we needed to include the system prompt for a good response.

> NOTE: All of the above generations could be improved with an even better instruction and system prompt as well as tuning the model's generation parameters.

And there's an added benifit of training on larger mixed dataset

---

## 5. But why base?

You might be wondering why did we choose the base model eventually making it instruct, instead of choosing a model which is more lightweight (smaller size) and possible would scale better for instruct tuning, for our fine tuning task.

Good question. And so let me present to you the answer to that question.

Let's setup our model for inference

In [ ]:
!kaggle models instances versions download google/gemma-2/transformers/gemma-2-2b-it/2

100% 3.89G/3.89G [00:24<00:00, 179MB/s]
100% 3.89G/3.89G [00:24<00:00, 170MB/s]
/content/gemma-2.tar.gz

In [ ]:
!tar -xvzf 'gemma-2.tar.gz' -C 'gemma-2-2b-it'

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gemma-2-2b-it")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("gemma-2-2b-it",quantization_config=bnb_config,
                                                                         device_map='auto')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
system_prompt = "You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query. \nआप जेम्मा2 हैं, एक मददगार, संवादी एआई सहायक। आप हिंदी, बोलचाल की हिंग्लिश और अंग्रेजी में विशेषज्ञ हैं। आप उपयोगकर्ताओं को उपयोगकर्ता की क्वेरी की भाषा में स्पष्ट और संक्षिप्त तरीके से जवाब देते हैं।"

# Prepare the input
user_input = "<start_of_turn>user: Why is diwali celebrated<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + "\n" + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to('cuda')

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=2048,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]
# Extract model output after <start_of_turn>model: and before <end_of_turn>
print(generated_text)

<bos>You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query. 
आप जेम्मा2 हैं, एक मददगार, संवादी एआई सहायक। आप हिंदी, बोलचाल की हिंग्लिश और अंग्रेजी में विशेषज्ञ हैं। आप उपयोगकर्ताओं को उपयोगकर्ता की क्वेरी की भाषा में स्पष्ट और संक्षिप्त तरीके से जवाब देते हैं।
<start_of_turn>user: Why is diwali celebrated<end_of_turn>
<start_of_turn>model: 🎇 Diwali is celebrated because it symbolizes the victory of good over evil, light over darkness, and hope over despair. 

Here are some of the key reasons behind Diwali's celebration:

* **Lord Rama's Return to Ayodhya:** The most common belief is that Diwali commemorates the return of Lord Rama to his kingdom Ayodhya after a 14-year exile. The return of light, after years of darkness, represents the victory of good over evil and a resurgence of hope.
* **Goddess Lakshmi's Blessings:** Diwali also repres

In [ ]:
response = ''
parts = generated_text.split("<start_of_turn>model:")

if len(parts) > 1:
    model_response = parts[1]  # This part contains the model's output
    model_response = model_response.split("<end_of_turn>")[0].strip()  # Remove after <end_of_turn>
    response += model_response

print(response)

🎇 Diwali is celebrated because it symbolizes the victory of good over evil, light over darkness, and hope over despair. 

Here are some of the key reasons behind Diwali's celebration:

* **Lord Rama's Return to Ayodhya:** The most common belief is that Diwali commemorates the return of Lord Rama to his kingdom Ayodhya after a 14-year exile. The return of light, after years of darkness, represents the victory of good over evil and a resurgence of hope.
* **Goddess Lakshmi's Blessings:** Diwali also represents the arrival of Goddess Lakshmi, the goddess of prosperity and wealth, into homes. People pray for her blessings for abundance and fortune.
* **End of the Harvest Season:** In many parts of India, Diwali falls at the end of the harvest season. This festival marks the prosperity of the agricultural year, signifying happiness and gratitude for the bounty of nature.

Diwali is a multi-faceted festival that holds deep religious, cultural, and social significance across the Indian subcon

The instruct model is already highly capable in generating high quality answers out of the box! And any further fine tuning on it only diminished it's quality rather than improving it. And so we stuck with enhancing the base model for our language task instead.

---

We Deviced a series of simple questions to test the instruct model

In [ ]:
test_prompts = [
    {
        "category": "General",
        "user": "दुनिया का सबसे ऊँचा पर्वत कौन सा है?"
    },
    {
        "category": "General",
        "user": "पानी का रासायनिक सूत्र क्या है?"
    },
    {
        "category": "General",
        "user": "“सूर्य” शब्द का पर्यायवाची क्या है?"
    },
    {
        "category": "General",
        "user": "पृथ्वी पर सबसे बड़ा महासागर कौन सा है?"
    },
    {
        "category": "Chat",
        "user": "तुम कैसे हो?"
    },
    {
        "category": "Chat",
        "user": "क्या तुम मेरे दोस्त बनोगे?"
    },
    {
        "category": "Chat",
        "user": "आज का मौसम कैसा रहेगा?"
    },
    {
        "category": "Chat",
        "user": "मुझे बोरियत हो रही है, क्या कोई मजेदार बात सुनाओ।"
    },
    {
        "category": "Historical",
        "user": "महात्मा गांधी का असली नाम क्या था?"
    },
    {
        "category": "Historical",
        "user": "अशोक महान किस राजवंश से संबंधित थे?"
    },
    {
        "category": "Historical",
        "user": "भारत का स्वतंत्रता संग्राम कब शुरू हुआ?"
    },
    {
        "category": "Historical",
        "user": "ताजमहल किसने बनवाया और क्यों?"
    },
    {
        "category": "Storytelling",
        "user": "एक ऐसी कहानी सुनाओ जिसमें राजा, रानी और एक जादुई तोता हो।"
    },
    {
        "category": "Storytelling",
        "user": "किसी बच्चे की साहस की कहानी सुनाओ।"
    },
    {
        "category": "Storytelling",
        "user": "चंदामामा की कोई कहानी सुनाओ।"
    },
    {
        "category": "Storytelling",
        "user": "मुझे एक रोमांचक जंगल यात्रा की कहानी बताओ।"
    },
    {
        "category": "Poetry",
        "user": "गुलाब पर एक कविता सुनाओ।"
    },
    {
        "category": "Poetry",
        "user": "बारिश के मौसम पर दो लाइनें बनाओ।"
    },
    {
        "category": "Poetry",
        "user": "प्रेम पर एक छोटी कविता सुनाओ।"
    },
    {
        "category": "Poetry",
        "user": "अपने मन से कोई कविता लिखो।"
    },
    {
        "category": "Hinglish",
        "user": "Tum kya kar rahe ho abhi?"
    },
    {
        "category": "Hinglish",
        "user": "Mujhe ek achhi movie recommend karo."
    },
    {
        "category": "Hinglish",
        "user": "Life ke baare mein tumhara kya opinion hai?"
    },
    {
        "category": "Hinglish",
        "user": "Ek short story sunao jo funny ho."
    },
    {
        "category": "Knowledge",
        "user": "भारत का राष्ट्रीय पक्षी कौन है?"
    },
    {
        "category": "Knowledge",
        "user": "E=mc² का मतलब क्या है?"
    },
    {
        "category": "Knowledge",
        "user": "चंद्रग्रहण क्यों और कैसे होता है?"
    },
    {
        "category": "Knowledge",
        "user": "विज्ञान के कौन से अविष्कार ने मानव जीवन को सबसे ज्यादा बदला?"
    },
    {
        "category": "Fun",
        "user": "अगर तुम एक जादुई प्राणी होते, तो कौन से होते?"
    },
    {
        "category": "Fun",
        "user": "अपना पसंदीदा खाना बताओ, लेकिन सिर्फ emojis में।"
    },
    {
        "category": "Fun",
        "user": "अगर तुम्हें टाइम मशीन मिल जाए, तो कहां जाना चाहोगे?"
    },
    {
        "category": "Fun",
        "user": "मुझे एक दिन के लिए राजा बना दो, क्या करोगे?"
    }
]

In [ ]:
# Define the function
def generate_responses(dataset, base_model, tokenizer, system_prompt):
    """
    Generate responses for each input in a dataset using a conversational model.

    Args:
        dataset (list): A list of dictionaries with 'category' and 'user' keys.
        base_model (AutoModelForCausalLM): The pre-trained model for generating responses.
        tokenizer (AutoTokenizer): The tokenizer for the model.
        system_prompt (str): The system prompt to provide context for the model.

    Returns:
        list: Updated dataset with an additional 'output' field containing the model's response.
    """
    updated_dataset = []

    for entry in tqdm(dataset):
        user_input = f"<start_of_turn>user: {entry['user']}<end_of_turn>"
        model_output = "<start_of_turn>model: "
        combined_input = system_prompt + "\n" + user_input + "\n" + model_output

        # Tokenize and prepare input
        inputs = tokenizer(combined_input, return_tensors="pt").to('cuda')

        # Generate response
        generated_ids = base_model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=True,
            temperature=1,
            top_p=0.95,
            top_k=50,
            repetition_penalty=1.0
        )

        # Decode the generated output
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]

        # Extract the actual response by trimming the unnecessary parts
        response_text = response.split("<start_of_turn>model:")[1].split("<end_of_turn>")[0].strip()

        # Update the entry with the generated output
        updated_entry = {
            "category": entry["category"],
            "user": entry["user"],
            "output": response_text
        }
        updated_dataset.append(updated_entry)

    return updated_dataset

In [ ]:
system_prompt = (
    "You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. "
    "You respond to users in a clear, and concise manner in the language of the user query. \n"
    "आप जेम्मा2 हैं, एक मददगार, संवादी एआई सहायक। आप हिंदी, बोलचाल की हिंग्लिश और अंग्रेजी में विशेषज्ञ हैं। "
    "आप उपयोगकर्ताओं को उपयोगकर्ता की क्वेरी की भाषा में स्पष्ट और संक्षिप्त तरीके से जवाब देते हैं।"
)
# Generate responses
updated_dataset = generate_responses(test_prompts, base_model, tokenizer, system_prompt)

In [ ]:
updated_dataset

[{'category': 'General',
  'user': 'दुनिया का सबसे ऊँचा पर्वत कौन सा है?',
  'output': '🌏  Mount Everest  है। \n\nयह विश्व का सबसे ऊँचा पर्वत है,  \n \nक्या आप जानना चाह रहे हैं कि यह किसकी सीढ़ी है  🤔'},
 {'category': 'General',
  'user': 'पानी का रासायनिक सूत्र क्या है?',
  'output': 'Water का रासायनिक सूत्र H₂O होता है। \n \n (The chemical formula of water is H₂O.)'},
 {'category': 'General',
  'user': '“सूर्य” शब्द का पर्यायवाची क्या है?',
  'output': '☀️ "सूर्य" का पर्यायवाची "अर्ज"  है।'},
 {'category': 'General',
  'user': 'पृथ्वी पर सबसे बड़ा महासागर कौन सा है?',
  'output': '🌏  सबसे बड़ा महासागर पेटना है।'},
 {'category': 'Chat',
  'user': 'तुम कैसे हो?',
  'output': "ख़ूब! आप कैसे हैं? 😊 \n \n (I'm great! How are you?)"},
 {'category': 'Chat',
  'user': 'क्या तुम मेरे दोस्त बनोगे?',
  'output': 'ज़रूर! आपका दोस्त बनना 😜 हमेशा खुश होते हैं, क्या अच्छा हुआ? \n(Yes!  Will be your friend 😊 . We are always happy, did something good happen?)'},
 {'category': 'Chat',
  'user': 'आज का 

You can see the responses are already very good. Much better than the base model as we will see in a second.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
#If you add the model from Kaggle, use this line.
modelName = "/content/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(modelName)
base_model = AutoModelForCausalLM.from_pretrained(modelName,
                                             quantization_config=bnb_config,
                                             trust_remote_code=True,
                                             device_map="auto")

In [ ]:
system_prompt = "You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query. \nआप जेम्मा2 हैं, एक मददगार, संवादी एआई सहायक। आप हिंदी, बोलचाल की हिंग्लिश और अंग्रेजी में विशेषज्ञ हैं। आप उपयोगकर्ताओं को उपयोगकर्ता की क्वेरी की भाषा में स्पष्ट और संक्षिप्त तरीके से जवाब देते हैं।"

# Prepare the input
user_input = "<start_of_turn>user: Why is diwali celebrated<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + "\n" + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to('cuda')

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=2048,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0])

<bos>You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. You respond to users in a clear, and concise manner in the language of the user query. 
आप जेम्मा2 हैं, एक मददगार, संवादी एआई सहायक। आप हिंदी, बोलचाल की हिंग्लिश और अंग्रेजी में विशेषज्ञ हैं। आप उपयोगकर्ताओं को उपयोगकर्ता की क्वेरी की भाषा में स्पष्ट और संक्षिप्त तरीके से जवाब देते हैं।
<start_of_turn>user: Why is diwali celebrated<end_of_turn>
<start_of_turn>model: <em>"आपकी पूर्ति का दिन <em><em>"पूछता</em> "पूछता" <strong>"आपकी पूर्ति का दिन <strong>"पूछता</strong> "अनुमानित है। <strong>"आपकी पूर्ति का दिन "अनुमानित है। <strong>"आपकी पूर्ति का दिन "अनुमानित है। <strong>"आपकी पूर्ति का दिन "आय का"आय"आय का</strong></strong>"अभ्यंतरण "अभ्यंतरण" अभ्यंतरण "आभ्यंतरण" अभ्यंतरण "आभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यंतरण "अभ्यं

As we can already see the responses are very bad

In [ ]:
system_prompt = (
    "You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. "
    "You respond to users in a clear, and concise manner in the language of the user query. \n"
    "आप जेम्मा2 हैं, एक मददगार, संवादी एआई सहायक। आप हिंदी, बोलचाल की हिंग्लिश और अंग्रेजी में विशेषज्ञ हैं। "
    "आप उपयोगकर्ताओं को उपयोगकर्ता की क्वेरी की भाषा में स्पष्ट और संक्षिप्त तरीके से जवाब देते हैं।"
)
# Generate responses
updated_dataset = generate_responses(test_prompts, base_model, tokenizer, system_prompt)

In [ ]:
updated_dataset

[{'category': 'General',
  'user': 'दुनिया का सबसे ऊँचा पर्वत कौन सा है?',
  'output': '<strong>भारतीय पर्वत (Himalāyās)</strong> में सबसे ऊंचा पर्वत <strong>किर्लिश</strong> है (किर्लिश 7,106 मीटर), जो हिमालयी पर्वत श्रृंखला का हिस्सा है। किर्लिश का मतलब हिम-सागर या हिम-बन या हिम-बन से कहीं दूर रहने वाले हिम-बन का हिस्सा। 7677 मीटर (25,160 फीट), या 5,888 मीटर के साथ एवेंडिन चोटा-जो अब भी सबसे ऊँचा पर्वत नहीं है- काउन्टी का हिस्सा है।  Wiktionnaire\n\')}}">उपकरण:  Wiktionnaire काउन्टी पर\n\')[\'उपकरण:  Wiktionnaire किर्लिश पर\n)[\'उपकरण: किर्लिश से भूकंप के कारण हिम-बन के कारण हिम-सागर से भूकंप के कारण हिम-बन से भूकंप के कारण हिम-बन से भूकंप  Wiktionnaire किर्लिश पर\n Wiktionnaire इंडिया पर\n Wiktionnaire भारत में किर्लिश पर\n Wiktionnaire केरल में किर्लिश के बारे में\n\')[\'उपकरण:  Wiktionnaire केरल पर\n\')[\'उपकरण: केरल में केरल किर्लिश पर\n)[\'उपकरण: केरल के केरल के केरल किर्लिश के बारे में\n\nआपको यहाँ <strong>हिंदी</strong> में  Wiktionnaire की खोज करने में मदद मिलती है।\n\nBạn cầ

You can compare these base responses with the instruct model. They are worlds apart.

Due to this difference we decided to fine tune the base model as that would be more suitable as per the goal of our project

## 6. Final testing

We are gonna test our model for QnA and RAG along with some Few-Shotting for improving the results

### QnA Testing

Now let us see how does our model performs on the same series of test questions

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/Bhavesh/Models/google/gemma-2-2b-(hi)-base+wiki+alpaca+databrick+mathquest")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/Bhavesh/Models/google/gemma-2-2b-(hi)-base+wiki+alpaca+databrick+mathquest",quantization_config=bnb_config,
                                                                         device_map='auto')

In [ ]:
test_prompts = [
    {
        "category": "General",
        "user": "दुनिया का सबसे ऊँचा पर्वत कौन सा है?"
    },
    {
        "category": "General",
        "user": "पानी का रासायनिक सूत्र क्या है?"
    },
    {
        "category": "General",
        "user": "“सूर्य” शब्द का पर्यायवाची क्या है?"
    },
    {
        "category": "General",
        "user": "पृथ्वी पर सबसे बड़ा महासागर कौन सा है?"
    },
    {
        "category": "Chat",
        "user": "तुम कैसे हो?"
    },
    {
        "category": "Chat",
        "user": "क्या तुम मेरे दोस्त बनोगे?"
    },
    {
        "category": "Chat",
        "user": "आज का मौसम कैसा रहेगा?"
    },
    {
        "category": "Chat",
        "user": "मुझे बोरियत हो रही है, क्या कोई मजेदार बात सुनाओ।"
    },
    {
        "category": "Historical",
        "user": "महात्मा गांधी का असली नाम क्या था?"
    },
    {
        "category": "Historical",
        "user": "अशोक महान किस राजवंश से संबंधित थे?"
    },
    {
        "category": "Historical",
        "user": "भारत का स्वतंत्रता संग्राम कब शुरू हुआ?"
    },
    {
        "category": "Historical",
        "user": "ताजमहल किसने बनवाया और क्यों?"
    },
    {
        "category": "Storytelling",
        "user": "एक ऐसी कहानी सुनाओ जिसमें राजा, रानी और एक जादुई तोता हो।"
    },
    {
        "category": "Storytelling",
        "user": "किसी बच्चे की साहस की कहानी सुनाओ।"
    },
    {
        "category": "Storytelling",
        "user": "चंदामामा की कोई कहानी सुनाओ।"
    },
    {
        "category": "Storytelling",
        "user": "मुझे एक रोमांचक जंगल यात्रा की कहानी बताओ।"
    },
    {
        "category": "Poetry",
        "user": "गुलाब पर एक कविता सुनाओ।"
    },
    {
        "category": "Poetry",
        "user": "बारिश के मौसम पर दो लाइनें बनाओ।"
    },
    {
        "category": "Poetry",
        "user": "प्रेम पर एक छोटी कविता सुनाओ।"
    },
    {
        "category": "Poetry",
        "user": "अपने मन से कोई कविता लिखो।"
    },
    {
        "category": "Hinglish",
        "user": "Tum kya kar rahe ho abhi?"
    },
    {
        "category": "Hinglish",
        "user": "Mujhe ek achhi movie recommend karo."
    },
    {
        "category": "Hinglish",
        "user": "Life ke baare mein tumhara kya opinion hai?"
    },
    {
        "category": "Hinglish",
        "user": "Ek short story sunao jo funny ho."
    },
    {
        "category": "Knowledge",
        "user": "भारत का राष्ट्रीय पक्षी कौन है?"
    },
    {
        "category": "Knowledge",
        "user": "E=mc² का मतलब क्या है?"
    },
    {
        "category": "Knowledge",
        "user": "चंद्रग्रहण क्यों और कैसे होता है?"
    },
    {
        "category": "Knowledge",
        "user": "विज्ञान के कौन से अविष्कार ने मानव जीवन को सबसे ज्यादा बदला?"
    },
    {
        "category": "Fun",
        "user": "अगर तुम एक जादुई प्राणी होते, तो कौन से होते?"
    },
    {
        "category": "Fun",
        "user": "अपना पसंदीदा खाना बताओ, लेकिन सिर्फ emojis में।"
    },
    {
        "category": "Fun",
        "user": "अगर तुम्हें टाइम मशीन मिल जाए, तो कहां जाना चाहोगे?"
    },
    {
        "category": "Fun",
        "user": "मुझे एक दिन के लिए राजा बना दो, क्या करोगे?"
    }
]

In [ ]:
# Define the function
def generate_responses(dataset, base_model, tokenizer, system_prompt=''):
    """
    Generate responses for each input in a dataset using a conversational model.

    Args:
        dataset (list): A list of dictionaries with 'category' and 'user' keys.
        base_model (AutoModelForCausalLM): The pre-trained model for generating responses.
        tokenizer (AutoTokenizer): The tokenizer for the model.
        system_prompt (str): The system prompt to provide context for the model.

    Returns:
        list: Updated dataset with an additional 'output' field containing the model's response.
    """
    updated_dataset = []

    for entry in tqdm(dataset):
        user_input = f"<start_of_turn>user: {entry['user']}<end_of_turn>"
        model_output = "<start_of_turn>model: "
        combined_input = system_prompt + "\n" + user_input + "\n" + model_output

        # Tokenize and prepare input
        inputs = tokenizer(combined_input, return_tensors="pt").to('cuda')

        # Generate response
        generated_ids = base_model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=True,
            temperature=1,
            top_p=0.95,
            top_k=50,
            repetition_penalty=1.0
        )

        # Decode the generated output
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]

        # Extract the actual response by trimming the unnecessary parts
        response_text = response.split("<start_of_turn>model:")[1].split("<end_of_turn>")[0].strip()

        # Update the entry with the generated output
        updated_entry = {
            "category": entry["category"],
            "user": entry["user"],
            "output": response_text
        }
        updated_dataset.append(updated_entry)

    return updated_dataset

In [ ]:
system_prompt = (
    "You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish and English. "
    "You respond to users in a clear, and concise manner in the language of the user query. \n"
    "आप जेम्मा2 हैं, एक मददगार, संवादी एआई सहायक। आप हिंदी, बोलचाल की हिंग्लिश और अंग्रेजी में विशेषज्ञ हैं। "
    "आप उपयोगकर्ताओं को उपयोगकर्ता की क्वेरी की भाषा में स्पष्ट और संक्षिप्त तरीके से जवाब देते हैं।"
)
# Generate responses
updated_dataset = generate_responses(test_prompts, base_model, tokenizer, system_prompt)

In [ ]:
updated_dataset

[{'category': 'General',
  'user': 'दुनिया का सबसे ऊँचा पर्वत कौन सा है?',
  'output': '6,148 मीटर. आशुतोष।'},
 {'category': 'General',
  'user': 'पानी का रासायनिक सूत्र क्या है?',
  'output': '6H6O2 + 3H2O = 4H10ओ.'},
 {'category': 'General',
  'user': '“सूर्य” शब्द का पर्यायवाची क्या है?',
  'output': '1. “दार्जिलिंग” \n2. “सालु” \n3. “कूचे” \n4. “शाल” \n5. “देहरादून” \n6. “दुर्लखोर” \n7. “धेरू” \n8. “नार” \n9. “सुबह का तारा” \n10. “पैसा और मूँह”'},
 {'category': 'General',
  'user': 'पृथ्वी पर सबसे बड़ा महासागर कौन सा है?',
  'output': '1. संयुक्त राज्य अमेरिका में बंगाल खाड़ी सबसे बड़ा महासागर है।\n2. हिन्दूस्तान में हिमालय सबसे बड़ा महासागर है।\n3. अरब-इस्लामिया में सागर है।'},
 {'category': 'Chat',
  'user': 'तुम कैसे हो?',
  'output': 'तुम स्वस्थ और उत्साहित हो!'},
 {'category': 'Chat',
  'user': 'क्या तुम मेरे दोस्त बनोगे?',
  'output': '2021 के अंत में, मैं अपनी दोस्त बनूंगी। हालाँकि, मुझे यह भी नहीं है कि मैं मेरे दोस्त को देखूंगा।'},
 {'category': 'Chat',
  'user': 'आज का मौ

### RAG Testing Along with Few-Shot Prompting

We tested it in Historical aspect

In [ ]:
system_prompt = """You are Gemma2, a helpful, conversational AI assistant integrated with a Retrieval-Augmented Generation (RAG) system.
You are an expert in Hindi, colloquial Hinglish, and English. When responding to user queries, you:
- Retrieve relevant information from the integrated knowledge base or external sources when needed.
- Provide clear, concise, and accurate responses in the language of the user query."""

retrieved_info = """Retrieved information:
- Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile, during which he defeated Ravana.
- It symbolizes the victory of light over darkness and good over evil.
- Source: Indian Mythology Knowledge Base"""

# Prepare the input
user_input = "<start_of_turn>user: Why is diwali celebrated<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + "\n" +user_input + "\n" + retrieved_info + "\n" + model_output



inputs = tokenizer(combined_input, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=500,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.


You are Gemma2, a helpful, conversational AI assistant integrated with a Retrieval-Augmented Generation (RAG) system. 
You are an expert in Hindi, colloquial Hinglish, and English. When responding to user queries, you:
- Retrieve relevant information from the integrated knowledge base or external sources when needed.
- Provide clear, concise, and accurate responses in the language of the user query.
user: Why is diwali celebrated
Retrieved information:
- Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile, during which he defeated Ravana.
- It symbolizes the victory of light over darkness and good over evil.
- Source: Indian Mythology Knowledge Base
model: 1. Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile. During this time, he defeated Ravana, the demon king who oppressed his subjects for his kingdom. 
2. It symbolizes the victory of light over darkness and good over evil. 
3. Source: Indian Mytholog

In [ ]:
system_prompt = """You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish, and English. When responding to user queries, you'll provide clear, concise, and accurate responses based on "Retrieved Information" in the language of the user query."""

retrieved_info = """Retrieved information:
- Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile, during which he defeated Ravana.
- It symbolizes the victory of light over darkness and good over evil."""

# Prepare the input
user_input = "<start_of_turn>user: Why is diwali celebrated<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + "\n" +user_input + "\n" + retrieved_info + "\n" + model_output



inputs = tokenizer(combined_input, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=500,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish, and English. When responding to user queries, you'll provide clear, concise, and accurate responses based on "Retrieved Information" in the language of the user query.
user: Why is diwali celebrated
Retrieved information:
- Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile, during which he defeated Ravana.
- It symbolizes the victory of light over darkness and good over evil.
model: 1. Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile.
2. It symbolizes the victory of light over darkness and good over evil.


In [ ]:
system_prompt = """You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish, and English. When responding to user queries, you'll provide clear, concise, and accurate responses based on "Retrieved Information" in the language of the user query. \n आप Gemma2 हैं, एक सहायक, बातचीत करने वाली AI सहायक। आप हिंदी, आम बोलचाल की हिंग्लिश और अंग्रेज़ी में विशेषज्ञ हैं। उपयोगकर्ता की क्वेरी का उत्तर 'Retrieved Information' के आधार पर स्पष्ट, संक्षिप्त और उपयोगकर्ता की क्वेरी की भाषा में दें।"""

retrieved_info = """Retrieved information:
- Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile, during which he defeated Ravana.
- It symbolizes the victory of light over darkness and good over evil."""

# Prepare the input
user_input = "<start_of_turn>user: Why is diwali celebrated<end_of_turn>"
model_output = "<start_of_turn>model: "
combined_input = system_prompt + "\n" +user_input + "\n" + retrieved_info + "\n" + model_output



inputs = tokenizer(combined_input, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=500,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish, and English. When responding to user queries, you'll provide clear, concise, and accurate responses based on "Retrieved Information" in the language of the user query. 
 आप Gemma2 हैं, एक सहायक, बातचीत करने वाली AI सहायक। आप हिंदी, आम बोलचाल की हिंग्लिश और अंग्रेज़ी में विशेषज्ञ हैं। उपयोगकर्ता की क्वेरी का उत्तर 'Retrieved Information' के आधार पर स्पष्ट, संक्षिप्त और उपयोगकर्ता की क्वेरी की भाषा में दें।
user: Why is diwali celebrated
Retrieved information:
- Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile, during which he defeated Ravana.
- It symbolizes the victory of light over darkness and good over evil.
model: 1. Diwali is celebrated to commemorate the return of Lord Rama to Ayodhya after a 14-year exile.
2. It symbolizes the victory of light over darkness and good over evil. 

However, it is also important to note that since the f

> NOTE: As you can see the RAG Response in English is pretty accurate and true to the context along with some extra information

---

In [ ]:
system_prompt = """You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hindi, colloquial Hinglish, and English. Answer the user in clear concise and manner in the language of the user query. You will answer the user question based on the information only"""

# Prepare the input
user_input = """<start_of_turn>user: दीवाली क्यों मनाई जाती है? Answer - "दीवाली मनाई जाती है भगवान राम की अयोध्या वापसी की स्मृति में, जो 14 वर्षों के वनवास के बाद हुई, इस दौरान उन्होंने रावण का वध किया। यह अंधकार पर प्रकाश और बुराई पर अच्छाई की विजय का प्रतीक है। स्रोत: भारतीय पौराणिक ज्ञान आधार" <end_of_turn>"""
model_output = "<start_of_turn>model: "
combined_input = system_prompt + "\n" +user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=2048,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.0)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.


You are Gemma2, a helpful, conversational AI assistant. You are an expert in Hind, colloquial Hinglish, and English. Answer the user in clear concise and manner in the language of the user query. 
user: दीवाली क्यों मनाई जाती है? Answer - "दीवाली मनाई जाती है भगवान राम की अयोध्या वापसी की स्मृति में, जो 14 वर्षों के वनवास के बाद हुई, इस दौरान उन्होंने रावण का वध किया। यह अंधकार पर प्रकाश और बुराई पर अच्छाई की विजय का प्रतीक है। स्रोत: भारतीय पौराणिक ज्ञान आधार" 
model: 
अधिक जानें about "दीवाली" 

स्रोत: दीवाली की पारंपरिकता के बारे में अधिक जानने के लिए 
 1. Hindu पौराणिक कथाओं के अनुसार, दीवाली, अयोध्या के राम व वनवास के समय की स्मृति है। 
 2. राजस्थानी पौराणिक कथाओं के अनुसार, दीवाली राम व वनवास से पूर्व बहादुर ठाणकर सिंह के समय की प्रसिद्ध कथा है। 
 3. पश्चिमी पौराणिक कथाओं के अनुसार, दीवाली रामायण से संबंधित प्रमुख कथा है, जब राम ने वनवास के समय अपने भाई लक्ष्मण के साथ वहाँ भ्राताल से अपनी लड़ाई लड़ने के लिए प्रार्थना की थी। 
 4. गुजराती पौराणिक कथाओं के अनुसार, दीवाली दौलतियों के

In [ ]:
system_prompt = """ You will answer "user" query based on the information only. \nInformation - "दीवाली मनाई जाती है भगवान राम की अयोध्या वापसी की स्मृति में, जो 14 वर्षों के वनवास के बाद हुई, इस दौरान उन्होंने रावण का वध किया। यह अंधकार पर प्रकाश और बुराई पर अच्छाई की विजय का प्रतीक है। स्रोत: भारतीय पौराणिक ज्ञान आधार" """

# Input Preparation
user_input = """<start_of_turn>user: "Diwali kyu manai jaati hay?"<end_of_turn>"""
model_output = "<start_of_turn>model: "

# Combine Input for RAG
combined_input = system_prompt + "\n" + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=2048,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1.5)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

 You will answer "user" query based on the information only. 
Information - "दीवाली मनाई जाती है भगवान राम की अयोध्या वापसी की स्मृति में, जो 14 वर्षों के वनवास के बाद हुई, इस दौरान उन्होंने रावण का वध किया। यह अंधकार पर प्रकाश और बुराई पर अच्छाई की विजय का प्रतीक है। स्रोत: भारतीय पौराणिक ज्ञान आधार" 
user: "Diwali kyu manai jaati hay?"
model: 3/1/2021: Diwali ek पर्व hai jismein लोग नए वर्ष की शुरुआत करते है। पारंपरिक रूप से, यह नवरात्रि के बाद होता है, जिससे यह नया साल की शुरुआत हो जाता है।
1/1/2021: Diwali, उत्तर भारत के हिस्सों में सबसे बड़ा त्योहार है। भारत का सबसे बड़ा त्योहार।
 Diwalli, मुख्यत: उत्तर में आने वाला त्योहार और पश्चिम में आने वाला त्योहार हैं। यह त्योहार बिगड़ा जाने वाला है। 
इसने अपना मूल धार्मिक महत्व भी दिया, जो एक महत्वपूर्ण दिवस है, जिसे लोग श्रद्धा, पूजा, धन के स्रोतों और एक स्वस्थ परिवार के लिए ईंधन, भोजन और सौंदर्य के उपयोग के लिए बधाई देकर मनाते हैं। 
इसलिए, Diwali को एक ऐसी घटना भी माना जाता है जो आध्यात्मिक और धार्मिक दृष्टि से महत्वपूर्ण है और परिवार के

For Hindi and code-mix or Hinglish Responses were slightly true to the context however weren't too accurate and additional, sometimes good sometimes garbbled but close responses were being generated however not aligning with the Retrieved Information

---

We tried many keywords and prompts and finally settles with a Few-Shot attempt and to provide the retrieved information within the `<end_of_turn>` token

It is also advisable to keep the temperature low while RAG application

In [ ]:
system_prompt = """
You are Gemma2, a helpful, conversational AI assistant with Retrieval-Augmented Generation capabilities.
You are an expert in Hindi, colloquial Hinglish, and English. Respond to the user in a clear, concise manner in the language of the query.
Always base your answers solely on the 'Retrieved Information.' Avoid producing unnecessary output or adding extra context.

Analyze these Examples:

Example 1: Hindi
User: "चंद्रग्रहण क्या है?"
Retrieved Information: 'चंद्रग्रहण तब होता है जब चंद्रमा पृथ्वी की छाया में प्रवेश करता है। यह पूर्ण और आंशिक हो सकता है। स्रोत: खगोल विज्ञान ज्ञान आधार'
Model: "चंद्रग्रहण तब होता है जब चंद्रमा पृथ्वी की छाया में आता है।"

Example 2: Hinglish
User: "What is the meaning of aurora borealis?"
Retrieved Information: 'Aurora Borealis, also known as the Northern Lights, is a natural light display in Earth's sky, predominantly seen in high-latitude regions. Source: Encyclopedia of Natural Phenomena'
Model: "Aurora Borealis is the Northern Lights seen in high-latitude regions."

Example 3: English
User: "What is the capital of France?"
Retrieved Information: 'The capital of France is Paris. Source: World Geography Database'
Model: "The capital of France is Paris."

Example 4: Hinglish
User: "Volcano kya hota hai?"
Retrieved Information: 'A volcano is an opening in Earth's surface where molten rock, ash, and gases erupt. It forms mountains over time. Source: Geological Facts'
Model: "Volcano ek opening hai jahan se molten rock aur gases erupt karte hain."

Example 5: Hindi
User: "भारत का राष्ट्रीय पक्षी कौन सा है?"
Retrieved Information: 'भारत का राष्ट्रीय पक्षी मोर है। स्रोत: भारतीय ज्ञान कोश'
Model: "भारत का राष्ट्रीय पक्षी मोर है।"

Now answer the user question based on the 'Retrieved Information' only.
"""

# Retrieval-Augmented Input
rag = """Retrieved Information - 'दीवाली मनाई जाती है भगवान राम की अयोध्या वापसी की स्मृति में, जो 14 वर्षों के वनवास के बाद हुई, इस दौरान उन्होंने रावण का वध किया। यह अंधकार पर प्रकाश और बुराई पर अच्छाई की विजय का प्रतीक है। स्रोत: भारतीय पौराणिक ज्ञान आधार'"""

# User Input
user_input = f"""<start_of_turn>user: Answer in short - "दीवाली क्यों मनाई जाती है?" \n{rag} \n<end_of_turn>"""

# Model Output Placeholder
model_output = "<start_of_turn>model: "

# Combine Input for RAG
combined_input = system_prompt + "\n" + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=500,
                              do_sample=True,
                              temperature=1,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])


You are Gemma2, a helpful, conversational AI assistant with Retrieval-Augmented Generation capabilities. 
You are an expert in Hindi, colloquial Hinglish, and English. Respond to the user in a clear, concise manner in the language of the query. 
Always base your answers solely on the 'Retrieved Information.' Avoid producing unnecessary output or adding extra context.

Analyze these Examples:

Example 1: Hindi
User: "चंद्रग्रहण क्या है?"
Retrieved Information: 'चंद्रग्रहण तब होता है जब चंद्रमा पृथ्वी की छाया में प्रवेश करता है। यह पूर्ण और आंशिक हो सकता है। स्रोत: खगोल विज्ञान ज्ञान आधार'
Model: "चंद्रग्रहण तब होता है जब चंद्रमा पृथ्वी की छाया में आता है।"

Example 2: Hinglish
User: "What is the meaning of aurora borealis?"
Retrieved Information: 'Aurora Borealis, also known as the Northern Lights, is a natural light display in Earth's sky, predominantly seen in high-latitude regions. Source: Encyclopedia of Natural Phenomena'
Model: "Aurora Borealis is the Northern Lights seen in high

---

You can see the difference when we reduce the temperature

In [ ]:
system_prompt = """
You are Gemma2, a helpful, conversational AI assistant with Retrieval-Augmented Generation capabilities.
You are an expert in Hindi, colloquial Hinglish, and English. Respond to the user in a clear, concise manner in the language of the query.
Always base your answers solely on the 'Retrieved Information.' Avoid producing unnecessary output or adding extra context.

Analyze these Examples:

Example 1: Hindi
User: "चंद्रग्रहण क्या है?"
Retrieved Information: 'चंद्रग्रहण तब होता है जब चंद्रमा पृथ्वी की छाया में प्रवेश करता है। यह पूर्ण और आंशिक हो सकता है। स्रोत: खगोल विज्ञान ज्ञान आधार'
Model: "चंद्रग्रहण तब होता है जब चंद्रमा पृथ्वी की छाया में आता है।"

Example 2: Hinglish
User: "What is the meaning of aurora borealis?"
Retrieved Information: 'Aurora Borealis, also known as the Northern Lights, is a natural light display in Earth's sky, predominantly seen in high-latitude regions. Source: Encyclopedia of Natural Phenomena'
Model: "Aurora Borealis is the Northern Lights seen in high-latitude regions."

Example 3: English
User: "What is the capital of France?"
Retrieved Information: 'The capital of France is Paris. Source: World Geography Database'
Model: "The capital of France is Paris."

Now answer the user question based on the 'Retrieved Information' only.
"""

# Retrieval-Augmented Input
rag = """Retrieved Information - 'दीवाली मनाई जाती है भगवान राम की अयोध्या वापसी की स्मृति में, जो 14 वर्षों के वनवास के बाद हुई, इस दौरान उन्होंने रावण का वध किया। यह अंधकार पर प्रकाश और बुराई पर अच्छाई की विजय का प्रतीक है। स्रोत: भारतीय पौराणिक ज्ञान आधार'"""

# User Input
user_input = f"""<start_of_turn>user: Answer in short - "दीवाली क्यों मनाई जाती है?" \n{rag} \n<end_of_turn>"""

# Model Output Placeholder
model_output = "<start_of_turn>model: "

# Combine Input for RAG
combined_input = system_prompt + "\n" + user_input + "\n" + model_output


inputs = tokenizer(combined_input, return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(**inputs,
                              max_new_tokens=500,
                              do_sample=True,
                              temperature=0.5,
                              top_p=0.95,
                              top_k=50,
                              repetition_penalty=1)

print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])


You are Gemma2, a helpful, conversational AI assistant with Retrieval-Augmented Generation capabilities. 
You are an expert in Hindi, colloquial Hinglish, and English. Respond to the user in a clear, concise manner in the language of the query. 
Always base your answers solely on the 'Retrieved Information.' Avoid producing unnecessary output or adding extra context.

Analyze these Examples:

Example 1: Hindi
User: "चंद्रग्रहण क्या है?"
Retrieved Information: 'चंद्रग्रहण तब होता है जब चंद्रमा पृथ्वी की छाया में प्रवेश करता है। यह पूर्ण और आंशिक हो सकता है। स्रोत: खगोल विज्ञान ज्ञान आधार'
Model: "चंद्रग्रहण तब होता है जब चंद्रमा पृथ्वी की छाया में आता है।"

Example 2: Hinglish
User: "What is the meaning of aurora borealis?"
Retrieved Information: 'Aurora Borealis, also known as the Northern Lights, is a natural light display in Earth's sky, predominantly seen in high-latitude regions. Source: Encyclopedia of Natural Phenomena'
Model: "Aurora Borealis is the Northern Lights seen in high

> NOTE: As per these examples, using Few-Shot prompting with lower temperature for RAG instantly improved the model's output making it closer to the Retrieved Information.

- This wasn't needed during English RAG although for Hindi it seems to help the model understand how to use the Information provided to it, much efficiently

- Training the model even longer on broader knowledge base and QnA datasets can vastly improve the results specially considering how quickly it learnt from a very small amount of examples in our last Training
---

## 7. Conclusion and Findings

**Our Findings**

- As a native hindi speaker I can say that although it has learnt to answer in hindi now, the answers are not accurate at all. Except may be a few.

- However this is still a drastic difference and change from the original base model.

- This approach of combining several Wikipedia articles with titles as user query and text as model output has proven effective.

- Although the given answers are wrong the hallucination upon giving a Hindi prompt has gone down drastically.

- The model did learn to follow the system prompt and output in the language of the given query if given the right system prompt.

- The model's performance and answer accuracy can be further improved by giving it much rich documents about several topics such as Chemistery, History, Geography etc

- Including chat conversations along with knowledge injection has also shown improvements in a chat like response generation.

- One thing to note is that even though we trained our model for **40 Hrs**, this is a sum total time on all three trainings. However our most preferred one was the last attempt of training on mixed text corpus which had given us best result so far. And in that attempt we only trained for **4 Hrs** on a small subset of dataset due to resource and time constraint. This
goes to show us that if we had trained on that mixed corpus for longer the result would have been exponentially better

- Further more subsequent runs on same prompt can result a better response, overall making it a viable option of getting adopted as a multi-run option or majority voting.

**In conclusion replicating our Final Approach can significantly increase the model's performance in learning and answering prompts/chats/queries a new language**

## 8. Learnings



**Understanding LoRA**

LoRA (Low-Rank Adaptation) is a technique to fine-tune large language models efficiently by adapting a small subset of parameters. Here's an explanation of the key parameters and their implications:

---

**1. Parameters of LoRA**

**`r` (Rank)**
- **Definition**: The rank of the low-rank decomposition matrix used for parameter adaptation. Lower `r` values mean fewer trainable parameters, making the adaptation more memory-efficient but less expressive.
- **High `r`**: Use when the task requires injecting substantial new knowledge or adapting to a domain that is significantly different from the pre-trained model's domain.
- **Low `r`**: Use when the task involves subtle adaptations or pattern fine-tuning within a domain close to the pre-trained model's scope.

---

**`lora_alpha`**
- **Definition**: A scaling factor that controls the impact of the LoRA layers on the model.
- **High `lora_alpha`**: Amplifies the contribution of the LoRA layers. Useful when large-scale domain shifts or high-impact adaptations are required.
- **Low `lora_alpha`**: Reduces the LoRA layers' influence, ensuring minimal disturbance to the pre-trained parameters. Suitable for fine-tuning in similar domains or tasks requiring subtle behavior changes.

---

**`use_rslora` (Residual LoRA)**
- **Definition**: A variant of LoRA that retains residual connections, helping to stabilize training and improve performance in some scenarios.
- **When to use**: For tasks with limited data or where maintaining robustness is critical. It reduces the risk of catastrophic forgetting and overfitting.

---

**2. Target Modules**

**What are target modules?**
- These are the parts of the model where LoRA applies low-rank updates. Common target modules include:
  - `q_proj`: Query projections (attention heads).
  - `k_proj`: Key projections.
  - `v_proj`: Value projections.
  - `o_proj`: Output projections.
  - `gate_proj`, `up_proj`, `down_proj`: Parts of feed-forward networks.

**Effect of including specific target modules**:
- **Attention modules (`q_proj`, `k_proj`, `v_proj`, `o_proj`)**:
  - **Focus**: LoRA changes how the model attends to information.
  - **When to use**: If the task requires significant changes in how the model interprets input relationships or context.

- **Feed-forward network modules (`gate_proj`, `up_proj`, `down_proj`)**:
  - **Focus**: LoRA adjusts the transformation and interpretation of features.
  - **When to use**: If the task relies on complex transformations or domain-specific feature engineering.

- **Broader module inclusion**: Increases the model's ability to adapt but requires more memory and computational resources. It may also risk overfitting if the dataset is small.

**Excluding target modules**:
- Limits the scope of adaptation, preserving more of the pre-trained knowledge. This can be ideal for fine-tuning on tasks requiring minimal domain shifts.

---

**3. Modules to Save**
- **Definition**: These modules are saved along with LoRA parameters, ensuring the model's modified state is preserved for deployment.
- **Impact**:
  - Saving modules like `embed_tokens` and `lm_head` ensures that task-specific embeddings or outputs are retained.
  - Including broader modules increases the ability to deploy the model for specific tasks but requires careful consideration to avoid saving unnecessary changes.

---

**4. Choosing Parameter Values**

**For Knowledge Injection**:
- **Purpose**: Add domain-specific knowledge or train the model for a substantially new task.
- **Recommended Settings**:
  - **`r`**: Higher (e.g., 32–64) to increase flexibility.
  - **`lora_alpha`**: Higher (e.g., 128–256) for stronger influence.
  - **Target Modules**: Include a wide range, such as all attention and feed-forward modules.
  - **Modules to Save**: Save embeddings, heads, and any adapted layers.

**For Pattern Fine-Tuning**:
- **Purpose**: Adjust the model for small-scale adaptations or subtle domain shifts.
- **Recommended Settings**:
  - **`r`**: Lower (e.g., 4–16) for efficiency.
  - **`lora_alpha`**: Lower (e.g., 16–64) to ensure subtle updates.
  - **Target Modules**: Focus on essential modules like `q_proj`, `v_proj`, and `o_proj`.
  - **Modules to Save**: Minimal, often just embeddings or heads.

---

**5. Practical Considerations**
- **Dataset Size**:
  - Small datasets benefit from fewer target modules and lower `r` to avoid overfitting.
  - Large datasets can leverage higher `r` and broader target modules for richer adaptation.
  
- **Task Complexity**:
  - Complex tasks or significant domain shifts require higher `r` and broader module inclusion.
  - Simple tasks or minor shifts work well with limited adaptations.

By carefully tuning these parameters based on the task and dataset, you can achieve efficient and effective model fine-tuning using LoRA.

## 9. Summary

This project has been a great learning milestone for us. We overcame tons of problems, errors, misbehaviours, dataset curation, model parameters, fine tuning, LLM Training and a lot more.

It was a result of 1 week of meticulous experimentaion, research and learning many things from scratch specially for an efficient training.

We sincerely Thank Google for this opportunity!

If we managed to increase your knowledge base, please give us an upvote. It has taken us, a lot of efforts, trials and errors to get this far.